# MIDAS · Caso 2 — Explorador del modelo de datos Bronze

### Cuaderno guiado para entender las tablas, los campos y sus relaciones

---

**Para quién es esto.** Para cualquiera que llegue nuevo al proyecto MIDAS y necesite
entender de dónde salen los datos del Caso 2 (*variación significativa de consumo*).
No se asume conocimiento previo de Oracle FLEX ni del modelo de EPM.

**Qué hace este cuaderno.** Esta edición es **autocontenida**: incorpora las queries
reales del bundle y puede explorar Oracle directamente aunque el repo y las Bronze todavía
no estén desplegados en Databricks. La segunda mitad conserva la exploración opcional de
Unity Catalog para cuando las tablas estén disponibles.

**Recorrido guiado.**
1. Explica el **modelo mental** del negocio (cliente → contrato → servicio → periodo).
2. Enseña a **leer los nombres de columna de FLEX**, que parecen jeroglíficos hasta que
   conoces la regla.
3. Recorre **las 13 tablas Bronze una por una**: qué es, de dónde sale en Oracle,
   qué columnas importan y cómo se ve el dato real.
4. Muestra **cómo se unen las tablas entre sí**, con las llaves verificadas.
5. Traduce los **casos de uso del analista** a consultas ejecutables.
6. Corre unas **verificaciones críticas del pipeline** que conviene revisar antes de confiar
   en los datos.

**Es 100% de solo lectura.** No escribe, no borra, no modifica nada. Puedes ejecutarlo
completo sin miedo.

---

> ### Cómo usarlo
> - Ejecuta las celdas **en orden**: las primeras definen configuración y funciones auxiliares
>   que las siguientes necesitan.
> - Si una tabla todavía no existe en tu ambiente, el cuaderno **te avisa y sigue** — no se rompe.
> - Cada sección tiene una explicación en texto **antes** del código. Léela: el código sin el
>   contexto no enseña gran cosa.
> - Las celdas marcadas **🔍 EXPLORA** están pensadas para que las modifiques y juegues con ellas.

---
# 0. Preparación del entorno para Oracle

Esta sección reproduce dentro del propio cuaderno el conector utilizado por MIDAS:
**JayDeBeApi + JPype1 + ojdbc11**. No importa `midas.db`, no lee archivos del repo y no
requiere que el bundle esté desplegado en el Workspace.

Requisitos del cluster:

- Databricks Runtime con JDK 17 (recomendado: DBR 16.4).
- Acceso de red a Oracle por DNS y TCP.
- Permiso `READ VOLUME` sobre el JAR de Oracle.
- Permiso de lectura del secret scope configurado.
- Usuario Oracle con permisos exclusivamente de consulta sobre las tablas utilizadas.

> Todo el ejecutor aplica una barrera de solo lectura: únicamente admite sentencias que
> comiencen por `SELECT` o `WITH`. No crea tablas, no guarda Parquet y no escribe en
> Unity Catalog.


In [ ]:
# Instala las dos librerías en el entorno del notebook.
# Databricks trata esta llamada igual que una celda %pip.
get_ipython().run_line_magic("pip", "install JayDeBeApi JPype1")


In [ ]:
# El reinicio es necesario para que JPype/JayDeBeApi queden disponibles.
# Al usar "Run all", Databricks continúa con las celdas siguientes después del reinicio.
dbutils.library.restartPython()


## 0.1 Configuración y selección de la muestra

Los valores predeterminados provienen del notebook de exploración usado en desarrollo.
La contraseña nunca se imprime: se obtiene en tiempo de ejecución desde Databricks Secrets.

Puedes conducir la cadena de dos maneras:

1. Dejar vacías las anclas: se toma una orden pendiente, priorizando la actividad `993`.
2. Escribir una orden o las llaves técnicas conocidas. Las anclas manuales tienen precedencia.

El límite se aplica **por ejecución de query**. Las queries embebidas son las literales del
repo; para proteger el origen se envuelven externamente con `ROWNUM <= max_filas`.


In [ ]:
def _widget_text(nombre, valor, etiqueta=None):
    try:
        dbutils.widgets.text(nombre, valor, etiqueta or nombre)
    except Exception:
        pass

_widget_text("oracle_host", "epm-to34.corp.epm.com.co", "Oracle host")
_widget_text("oracle_port", "1521", "Oracle port")
_widget_text("oracle_service", "SFUAT", "Oracle service")
_widget_text("oracle_user", "SQL_EPMBOTPD05", "Oracle user")
_widget_text("oracle_secret_scope", "AZ-SecretScopeDBKS-EPM-NP-KV-DLLO", "Secret scope")
_widget_text("oracle_password_key", "AZ-SECRET-EPM-BOTPD05-FACTURACION-CTATECNICA", "Password key")
_widget_text(
    "oracle_jdbc_jar_path",
    "/Volumes/epm_datalake_vol_np/facturacion_vol/facturacion_bronze_vol/_oracle_client/ojdbc11-23.26.2.0.0.jar",
    "Ruta ojdbc11",
)
_widget_text("max_filas_oracle", "50", "Máximo de filas por query")
_widget_text("max_servicios_contrato", "8", "Máximo de SS hermanos")
_widget_text("preferir_actividad", "993", "Actividad preferida")
_widget_text("id_orden", "", "Orden específica (opcional)")
_widget_text("servicio_suscrito", "", "SS manual (opcional)")
_widget_text("contrato", "", "Contrato manual (opcional)")
_widget_text("instalacion", "", "Instalación manual (opcional)")
_widget_text("id_periodo_consumo", "", "Periodo consumo manual")
_widget_text("id_periodo_facturacion", "", "Periodo facturación manual")
_widget_text("tipo_consumo", "", "Código tipo consumo manual")

ORACLE_HOST = dbutils.widgets.get("oracle_host").strip()
ORACLE_PORT = int(dbutils.widgets.get("oracle_port").strip())
ORACLE_SERVICE = dbutils.widgets.get("oracle_service").strip()
ORACLE_USER = dbutils.widgets.get("oracle_user").strip()
ORACLE_SCOPE = dbutils.widgets.get("oracle_secret_scope").strip()
ORACLE_PASSWORD_KEY = dbutils.widgets.get("oracle_password_key").strip()
ORACLE_JAR = dbutils.widgets.get("oracle_jdbc_jar_path").strip()
MAX_FILAS_ORACLE = max(1, int(dbutils.widgets.get("max_filas_oracle") or "50"))
MAX_SERVICIOS_CONTRATO = max(1, int(dbutils.widgets.get("max_servicios_contrato") or "8"))
ACTIVIDAD_PREFERIDA = dbutils.widgets.get("preferir_actividad").strip()

def _entero_widget(nombre):
    valor = dbutils.widgets.get(nombre).strip()
    return int(valor) if valor else None

ANCLAS_MANUALES = {
    "id_orden": _entero_widget("id_orden"),
    "servicio_suscrito": _entero_widget("servicio_suscrito"),
    "contrato": _entero_widget("contrato"),
    "instalacion": _entero_widget("instalacion"),
    "id_periodo_consumo": _entero_widget("id_periodo_consumo"),
    "id_periodo_facturacion": _entero_widget("id_periodo_facturacion"),
    "tipo_consumo": _entero_widget("tipo_consumo"),
}

print(f"Oracle: {ORACLE_HOST}:{ORACLE_PORT}/{ORACLE_SERVICE} · user={ORACLE_USER}")
print(f"Límite por query: {MAX_FILAS_ORACLE} · máximo SS del contrato: {MAX_SERVICIOS_CONTRATO}")
print("Anclas manuales:", {k: v for k, v in ANCLAS_MANUALES.items() if v is not None} or "ninguna")


## 0.2 Conector JDBC autocontenido

Las funciones siguientes son una copia autocontenida del contrato técnico de
`src/midas/db/database.py`:

- convierte binds Oracle `:nombre` a parámetros JDBC `?`;
- convierte objetos Java a tipos Python;
- devuelve cada resultado como `pandas.DataFrame`;
- verifica DNS y TCP antes de autenticar;
- bloquea cualquier sentencia que no sea de lectura.


In [ ]:
import os
import re
import socket
from datetime import datetime
from decimal import Decimal, InvalidOperation

import pandas as pd
import jaydebeapi

ORACLE_DRIVER = "oracle.jdbc.OracleDriver"
ORACLE_URL = f"jdbc:oracle:thin:@{ORACLE_HOST}:{ORACLE_PORT}/{ORACLE_SERVICE}"
_CONEXION_ORACLE = None

def diagnosticar_red():
    ip = socket.gethostbyname(ORACLE_HOST)
    print(f"✅ DNS: {ORACLE_HOST} -> {ip}")
    with socket.create_connection((ORACLE_HOST, ORACLE_PORT), timeout=10):
        print(f"✅ TCP: {ORACLE_HOST}:{ORACLE_PORT}")

def conectar_oracle():
    global _CONEXION_ORACLE
    if _CONEXION_ORACLE is not None:
        return _CONEXION_ORACLE
    if not ORACLE_JAR or not os.path.exists(ORACLE_JAR):
        raise FileNotFoundError(
            f"No existe el JAR configurado: {ORACLE_JAR!r}. "
            "Verifica la ruta y el permiso READ VOLUME."
        )
    diagnosticar_red()
    password = dbutils.secrets.get(scope=ORACLE_SCOPE, key=ORACLE_PASSWORD_KEY)
    _CONEXION_ORACLE = jaydebeapi.connect(
        ORACLE_DRIVER, ORACLE_URL, [ORACLE_USER, password], ORACLE_JAR
    )
    print("✅ Conexión JDBC Oracle establecida (solo lectura).")
    return _CONEXION_ORACLE

def cerrar_oracle():
    global _CONEXION_ORACLE
    if _CONEXION_ORACLE is not None:
        _CONEXION_ORACLE.close()
        _CONEXION_ORACLE = None
        print("Conexión Oracle cerrada.")

def _sanitizar_parametro(valor):
    if valor is None or isinstance(valor, (bool, int, float, str)):
        return valor
    if hasattr(valor, "item"):
        return valor.item()
    if isinstance(valor, pd.Timestamp):
        return valor.to_pydatetime().strftime("%Y-%m-%d %H:%M:%S")
    if isinstance(valor, datetime):
        return valor.strftime("%Y-%m-%d %H:%M:%S")
    return str(valor)

def _preparar_binds(sql, parametros):
    if not parametros:
        return sql, []
    claves = sorted(parametros, key=len, reverse=True)
    patron = re.compile(r":(" + "|".join(re.escape(k) for k in claves) + r")\b")
    orden = []
    def reemplazar(match):
        orden.append(match.group(1))
        return "?"
    preparado = patron.sub(reemplazar, sql)
    return preparado, [_sanitizar_parametro(parametros[k]) for k in orden]

def _clase_java(valor):
    try:
        return str(valor.getClass().getName())
    except Exception:
        return type(valor).__name__

def _a_python(valor, escala):
    if valor is None or isinstance(valor, (str, int, float, bool, bytes, datetime)):
        return valor
    clase = _clase_java(valor)
    if any(t in clase for t in ("BigDecimal", "Double", "Float", "Long", "Integer", "Short")):
        texto = str(valor)
        try:
            decimal = Decimal(texto)
        except InvalidOperation:
            return float(texto)
        if escala == 0 and decimal == decimal.to_integral_value():
            return int(decimal)
        return float(decimal)
    if "Timestamp" in clase or "java.sql.Date" in clase or "TIMESTAMP" in clase:
        return pd.to_datetime(str(valor)).to_pydatetime()
    if "Clob" in clase or "CLOB" in clase:
        try:
            return str(valor.getSubString(1, int(valor.length())))
        except Exception:
            return str(valor)
    return str(valor)

def _inicio_sql(sql):
    limpio = re.sub(r"(?m)^\s*--.*$", "", sql).lstrip()
    return limpio.split(None, 1)[0].upper() if limpio else ""

def ejecutar_select(sql, parametros=None):
    if _inicio_sql(sql) not in {"SELECT", "WITH"}:
        raise ValueError("Bloqueado: el explorador solo permite SELECT o WITH.")
    conexion = conectar_oracle()
    preparado, valores = _preparar_binds(sql, parametros or {})
    cursor = conexion.cursor()
    try:
        cursor.execute(preparado, valores)
        descripcion = cursor.description or []
        columnas = [c[0] for c in descripcion]
        escalas = [c[5] for c in descripcion]
        filas = cursor.fetchall()
        datos = [
            [_a_python(valor, escalas[i]) for i, valor in enumerate(fila)]
            for fila in filas
        ]
        return pd.DataFrame(datos, columns=columnas)
    finally:
        cursor.close()

def limitar_sql(sql, limite=MAX_FILAS_ORACLE):
    sql_limpio = sql.strip().rstrip(";")
    return f"SELECT * FROM (\n{sql_limpio}\n) q_midas WHERE ROWNUM <= :__limite_midas"

def mostrar_pd(df, titulo=None):
    if titulo:
        print(f"\n{titulo}")
    if df is None or df.empty:
        print(f"  ⚪ Sin filas. Columnas devueltas: {list(df.columns) if df is not None else []}")
        return
    display(df)


## 0.3 Las 13 queries reales, embebidas

Este diccionario se generó copiando literalmente las constantes vigentes de
`src/midas/db/queries.py`. Por eso el notebook no necesita clonar, instalar ni importar
el proyecto.

La columna `query_key` del inventario coincide con el seed real de
`midas_control_cargas`.


In [ ]:
QUERIES_ORACLE = {
  "QUERY_ORDENES_PENDIENTES": "\r\n--ordenes_calidad_pendientes\r\nSELECT\r\n    oa.order_id id_orden,\r\n    oa.product_id servicio_suscrito,\r\n    (select p.address_id from pr_product p where p.product_id = oa.product_id) instalacion,\r\n    oa.SUBSCRIPTION_ID contrato,\r\n    o.created_date fecha_creacion,\r\n    (select items_id from GE_ITEMS where oa.activity_id = items_id) || ' - ' || (select description actividad from GE_ITEMS where oa.activity_id = items_id) actividad,\r\n    o.order_status_id || ' - ' || (select description from or_order_status where order_status_id = o.order_status_id) estado_orden,\r\n    oa.comment_ comentario_orden\r\nFROM or_order_activity oa, or_order o\r\nWHERE o.order_id = oa.order_id\r\nAND oa.task_type_id = 883\r\nAND o.order_status_id <> 12\r\nAND o.LEGALIZATION_DATE is null\r\n",
  "QUERY_DATOS_BASICOS": "\r\n--datos_basicos_producto\r\nSELECT\r\n    sesunuse servicio_suscrito,\r\n    sesususc contrato,\r\n    p.address_id instalacion,\r\n    (SELECT servcodi||'-'||servdesc FROM servicio WHERE servcodi = sesuserv) servicio,\r\n    to_char(sesufein, 'YYYY-MM-DD') fecha_instalacion,\r\n    to_char(sesufere, 'YYYY-MM-DD') fecha_retiro,\r\n    (select * from (select periodicity from pe_per_his_prod where product_id= sesunuse order by created_date desc) where rownum <= 1) periodicidad,\r\n    (select escocodi||'-'||escodesc from estacort where escocodi = sesuesco) Estado_Corte,\r\n    (select catecodi||'-'||catedesc from categori where catecodi = sesucate) Categoria,\r\n    (select sucacodi||'-'||sucadesc from subcateg where sucacate = sesucate and sucacodi = sesusuca) subcategoria,\r\n    sesucicl as ciclo,\r\n    (select plsucodi||'-'||plsudesc from plansusc where plsucodi = sesuplfa) Plan_Facturacion,\r\n    (select plsucodi||'-'||plsudesc from pr_product,plansusc where commercial_plan_id = plsucodi and product_id = sesunuse) plan_facturacion_pr_product,\r\n    subscriber_name||' '||SUBS_LAST_NAME Nombre_Cliente,\r\n    cl.IDENTIFICATION identificacion,\r\n    (select geograp_location_id||'-'||description from ge_geogra_location g where g.geograp_location_id = d.geograp_location_id) Localidad,\r\n    d.address_parsed direccion,\r\n    cadastral_id PAGINA,\r\n    (select sum(cucosacu) from cuencobr where cuconuse = sesunuse and cucosacu > 0) SALDO_PENDIENTE,\r\n    (select count(1) from cuencobr where cuconuse = sesunuse and cucofeve < sysdate and cucosacu > 0) CUENTAS_VENCIDAS,\r\n    (select sum(cucosacu) from cuencobr where cuconuse = sesunuse and cucofeve < sysdate and cucosacu > 0) SALDO_VENCIDO\r\nFROM servsusc s, pr_product p, ab_address d, suscripc c, ge_subscriber cl\r\nWHERE 1=1\r\nAND sesunuse = product_id\r\nAND c.susccodi = s.sesususc\r\nAND p.product_id = s.sesunuse\r\nAND p.address_id = d.address_id\r\nAND cl.subscriber_id = c.suscclie\r\nAND d.address_id = :address_id -- Parametro de busqueda\r\n",
  "QUERY_DATOS_LECTURA": "\r\n--datos_lecturas_producto\r\nWITH periodos as (\r\n    SELECT *\r\n    FROM (\r\n        SELECT ss, tipoCons,\r\n               pefacodi, pefacicl,\r\n               pefaano, pefames,\r\n               pecscons, pecsfeci, pecsfecf,\r\n               dense_rank() over(order by pecsfecf desc) posicion\r\n        FROM (\r\n            SELECT cosssesu ss, cosspecs perCons, cosstcon tipoCons\r\n            FROM conssesu\r\n            WHERE cosssesu = :servicio_suscrito -- Argumento de entrada (servicio suscrito)\r\n            AND cossmecc = 4\r\n            UNION\r\n            SELECT leemsesu ss, leempecs perCons, leemtcon tipoCons\r\n            FROM lectelme\r\n            WHERE leemsesu = :servicio_suscrito -- Argumento de entrada (servicio suscrito)\r\n            AND leemclec = 'F'\r\n        ) p,\r\n        perifact,\r\n        pericose\r\n        WHERE pecscons = perCons\r\n        AND pefacicl = pecscico\r\n        AND pecsfecf BETWEEN pefafimo AND pefaffmo\r\n    )\r\n    WHERE posicion <= 8 -- Cantidad de lecturas\r\n),\r\nlecturas as (\r\n    SELECT leempecs, leemelme, leemtcon, leemleto, leemlean,\r\n           leemoble, leemobsb, leemobsc, leemliin, leemlisu, leemfame,\r\n           (SELECT order_comment\r\n            FROM or_order_activity oa, or_order_comment oc\r\n            WHERE oa.order_activity_id = leemdocu AND oc.order_id = oa.order_id\r\n            AND oc.comment_type_id = 4002 AND rownum <= 1) alfanumerica\r\n    FROM periodos, lectelme\r\n    WHERE leemsesu = ss\r\n    AND leempecs = pecscons\r\n    AND leemclec = 'F'\r\n    AND leemtcon = tipoCons\r\n),\r\nconsumos as (\r\n    SELECT cosspecs, cosselme, cosstcon, sum(cosscoca) cosscoca\r\n    FROM periodos, conssesu\r\n    WHERE cosssesu = ss\r\n    AND cosspecs = pecscons\r\n    AND cossmecc = 4\r\n    AND cosstcon = tipoCons\r\n    GROUP BY cosspecs, cosselme, cosstcon\r\n),\r\ncruce_lecturas_consumos as (\r\n    SELECT lecturas.*, consumos.*\r\n    FROM lecturas FULL OUTER JOIN consumos\r\n    ON (leempecs = cosspecs AND leemtcon = cosstcon AND leemelme = cosselme)\r\n)\r\nSELECT\r\n    ss servicio_suscrito,\r\n    pecscons id_periodo_consumo,\r\n    pefacodi id_periodo_facturacion,\r\n    to_char(pecsfeci, 'YYYY-MM-DD') fecha_ini_consumo,\r\n    to_char(pecsfecf, 'YYYY-MM-DD') fecha_fin_consumo,\r\n    round(pecsfecf - pecsfeci) dias_consumo,\r\n    (SELECT tconcodi||'-'||tcondesc FROM tipocons t WHERE t.tconcodi = tipocons) Tipo_Consumo,\r\n    tipoCons,\r\n    (SELECT elmecodi FROM elemmedi WHERE (elmeidem = leemelme OR elmeidem = cosselme)) Medidor,\r\n    (SELECT valor FROM elemmedi, ge_items_seriado gis, ge_items gi, ge_items_tipo_atr gita, GE_items_tipo_at_val gitav\r\n     WHERE (elmeidem = leemelme OR elmeidem = cosselme) AND serie = elmecodi AND gis.items_id = gi.items_id AND gita.attribute_id = 5000058\r\n     AND gitav.id_items_seriado = gis.id_items_seriado AND gitav.id_items_tipo_atr = gita.id_items_tipo_atr AND gi.id_items_tipo = gita.id_items_tipo) constante,\r\n    (SELECT ELMENUDC FROM elemmedi WHERE (elmeidem = leemelme OR elmeidem = cosselme)) digitos_medidor,\r\n    leemlean lectura_anterior,\r\n    leemleto lectura_actual,\r\n    (leemleto - leemlean) * leemfame consumo_calculado,\r\n    cosscoca consumo_facturado,\r\n    leemliin limite_inferior,\r\n    leemlisu limite_superior,\r\n    (SELECT oblecodi||'-'||obledesc FROM obselect WHERE oblecodi = leemoble) Observacion_Lectura,\r\n    (SELECT oblecodi||'-'||obledesc FROM obselect WHERE oblecodi = leemobsb) Observacion_Lectura_2,\r\n    (SELECT oblecodi||'-'||obledesc FROM obselect WHERE oblecodi = leemobsc) Observacion_Lectura_3,\r\n    (SELECT decode(count(1),0,'',count(1)) FROM conssesu cpno WHERE cpno.cosssesu = ss AND cpno.cosstcon = tipoCons AND cpno.cosspefa = pefacodi AND cossmecc= 17) PNO\r\nFROM periodos, cruce_lecturas_consumos\r\nWHERE (leempecs = pecscons OR cosspecs = pecscons)\r\nAND (leemtcon = tipoCons OR cosstcon = tipoCons)\r\nORDER BY pecsfecf desc, tipocons\r\n",
  "QUERY_DATOS_CONSUMOS": "\r\n--datos_consumos_producto\r\nSELECT\r\n    --cosspefa periodo,\r\n    cosssesu servicio_suscrito,\r\n    (select pecscons from pericose where pecscons = cosspecs) id_periodo_consumo,\r\n    cosspefa id_periodo_facturacion,\r\n    (select pefaano from perifact where pefacodi = cosspefa) anio_facturacion,\r\n    (select pefames from perifact where pefacodi = cosspefa) mes_facturacion,\r\n    (select pefacicl from perifact where pefacodi = cosspefa) ciclo,\r\n    (select CIOPCICO from cm_cicloppr where CIOPSESU = cosssesu) ciclo_operativo,\r\n    to_char(cossfere, 'YYYY-MM-DD') fecha_registro,\r\n    (select mecccodi||'-'||meccdesc from mecacons where mecccodi = cossmecc) metodo_calculo,\r\n    (select tconcodi||'-'||tcondesc from tipocons t where t.tconcodi = cosstcon) Tipo_Consumo,\r\n    cosscoca consumo,\r\n    cossfufa funcion_calculo,\r\n    (select cavccodi||'-'||cavcdesc from calivaco where cavccodi = cosscavc) calificacion\r\nFROM conssesu\r\nWHERE cosssesu = :p_servicio_suscrito --{Argumento 1 - servicio_suscrito}\r\n  AND cosspecs = :p_id_periodo_consumo --{Argumento 2 - periodo de consumo}\r\nORDER BY cosselme, cossfere\r\n",
  "QUERY_ORDENES_CRITICA_PEVIA": "\r\n--datos_ordenes_previa_critica\r\nWITH periodo as (\r\n    SELECT min(pefacodi) pefacodi, min(pecscons) pecscons,\r\n           min(pefafimo) pefafimo, min(pefaffmo) pefaffmo\r\n    FROM perifact, pericose\r\n    WHERE pefacodi = :p_id_periodo_facturacion --{Argumento 1 - periodo de facturacion}\r\n      AND pecscico = pefacicl\r\n      AND pefapecs = pecscons\r\n)\r\nSELECT /*+ leading (critica) ... */\r\n    o.order_id id_orden,\r\n    orcrsesu servicio_suscrito,\r\n    (SELECT tconcodi||'-'||tcondesc FROM tipocons t WHERE t.tconcodi = orcrtico) tipo_consumo,\r\n    orcrpeco id_periodo_consumo,\r\n    (select tt.task_type_id||'-'||tt.description from or_task_type tt where tt.task_type_id = o.task_type_id) Tipo_Trabajo,\r\n    (select items_id||'-'||description from ge_items where items_id = oa.activity_id) Actividad,\r\n    to_char(o.created_date, 'YYYY-MM-DD HH24:MI:SS') fecha_creacion_orden,\r\n    to_char(legalization_date, 'YYYY-MM-DD HH24:MI:SS') fecha_legalizacion_orden,\r\n    (select os.order_status_id||'-'||os.description from or_order_status os where os.order_status_id = o.order_status_id ) estado,\r\n    (select name_\r\n     from or_order_stat_change osc, ge_person pe, sa_user u\r\n     where pe.user_id = u.user_id\r\n       and u.mask = osc.user_id\r\n       and o.order_id = osc.order_id\r\n       and 5 = osc.initial_status_id\r\n       and 8 = osc.final_status_id) analista_legaliza\r\nFROM cm_ordecrit critica, or_order o, or_order_activity oa, periodo\r\nWHERE orcrsesu = :p_servicio_suscrito --{Argumento 2 - servicio suscrito}\r\n  AND orcrtico = nvl(:p_tipo_consumo, orcrtico) --{Argumento 3 - tipo de consumo}\r\n  AND oa.product_id = orcrsesu\r\n  AND o.order_id = oa.order_id\r\n  AND oa.order_activity_id = orcracti\r\n  AND oa.activity_id = 102010\r\n  AND orcrpeco = nvl(pecscons, orcrpeco)\r\nUNION\r\n(\r\n    SELECT /*+ index(p IDX_PE_INVEST_CONSUM01) ... */\r\n        o.order_id id_orden_critica,\r\n        p.product_id servicio_suscrito,\r\n        (SELECT tconcodi||'-'||tcondesc FROM tipocons t WHERE t.tconcodi = consumption_type) tipo_consumo,\r\n        consumption_period id_periodo_consumo,\r\n        (select tt.task_type_id||'-'||tt.description from or_task_type tt where tt.task_type_id = o.task_type_id) Tipo_Trabajo_Orden,\r\n        (select items_id||'-'||description from ge_items where items_id = oa.activity_id) Actividad,\r\n        to_char(o.created_date, 'YYYY-MM-DD HH24:MI:SS') fecha_creacion_orden,\r\n        to_char(legalization_date, 'YYYY-MM-DD HH24:MI:SS') fecha_legalizacion_orden,\r\n        (select os.order_status_id||'-'||os.description from or_order_status os where os.order_status_id = o.order_status_id ) estado,\r\n        (select name_\r\n         from or_order_stat_change osc, ge_person pe, sa_user u\r\n         where pe.user_id = u.user_id\r\n           and u.mask = osc.user_id\r\n           and o.order_id = osc.order_id\r\n           and 5 = osc.initial_status_id\r\n           and 8 = osc.final_status_id) analista_legaliza\r\n    FROM PE_INVEST_CONSUM p,\r\n         or_order o,\r\n         or_order_activity oa,\r\n         periodo\r\n    WHERE p.product_id = :p_servicio_suscrito --{Argumento 4 -servicio suscrito}\r\n      AND consumption_type = nvl(:p_tipo_consumo, consumption_type) --{Argumento 5 - tipo de consumo}\r\n      AND p.consumption_period = nvl(pecscons, p.consumption_period)\r\n      AND oa.package_id = p.investigate_request\r\n      AND oa.product_id = p.product_id\r\n      AND o.order_id = oa.order_id\r\n      AND oa.task_type_id in (769,803,807,10037,767,768,769,770,803,804,805,747,748,749,750,751,752,764,778,781) -- (Lista de IDs)\r\n    UNION\r\n    SELECT /*+ index(p IDX_PE_INVEST_CONSUM01) ... */\r\n        o.order_id id_orden,\r\n        p.product_id servicio_suscrito,\r\n        (SELECT tconcodi||'-'||tcondesc FROM tipocons t WHERE t.tconcodi = consumption_type) tipo_consumo,\r\n        consumption_period id_periodo_consumo,\r\n        (select tt.task_type_id||'-'||tt.description from or_task_type tt where tt.task_type_id = o.task_type_id) Tipo_Trabajo_Orden,\r\n        (select items_id||'-'||description from ge_items where items_id = oa.activity_id) Actividad,\r\n        to_char(o.created_date, 'YYYY-MM-DD HH24:MI:SS') fecha_creacion_orden,\r\n        to_char(legalization_date, 'YYYY-MM-DD HH24:MI:SS') fecha_legalizacion_orden,\r\n        (select os.order_status_id||'-'||os.description from or_order_status os where os.order_status_id = o.order_status_id ) estado,\r\n        (select name_\r\n         from or_order_stat_change osc, ge_person pe, sa_user u\r\n         where pe.user_id = u.user_id\r\n           and u.mask = osc.user_id\r\n           and o.order_id = osc.order_id\r\n           and 5 = osc.initial_status_id\r\n           and 8 = osc.final_status_id) analista_legaliza\r\n    FROM PE_INVEST_CONSUM p,\r\n         or_order o,\r\n         or_order_activity oa,\r\n         periodo\r\n    WHERE p.product_id = :p_servicio_suscrito --{Argumento 6 -servicio suscrito}\r\n      AND consumption_type = nvl(:p_tipo_consumo, consumption_type) --{Argumento 7 tipo de consumo}\r\n      AND p.register_date between pefafimo and pefaffmo\r\n      AND oa.package_id = p.investigate_request\r\n      AND oa.product_id = p.product_id\r\n      AND o.order_id = oa.order_id\r\n      AND oa.task_type_id in (769,803,807,10037,767,768,769,770,803,804,805,747,748,749,750,751,752,764,778,781) -- (Lista de IDs)\r\n)\r\nORDER BY fecha_creacion_orden desc\r\n",
  "QUERY_COMENTARIOS_ORDENES": "\r\n--datos_comentarios_ordenes\r\nSELECT\r\n    oc.order_id id_orden,\r\n    (select product_id from or_order_activity where order_id = oc.order_id  and rownum = 1) servicio_suscrito,\r\n    oc.register_date fecha_registro,\r\n    (select description from ge_comment_type ct where ct.comment_type_id = oc.comment_type_id) tipo_comentario,\r\n    replace(replace(replace(replace(oc.order_comment,chr(10), ''), chr(13), ''),chr(9),''),'|','') comentario\r\nFROM or_order_comment oc\r\nWHERE oc.order_id = :p_id_orden --{Argumento 1 - id_orden}\r\nUNION ALL\r\nSELECT distinct\r\n    :p_id_orden,\r\n    product_id,\r\n    adjustment_date,\r\n    'REVISION ANALISTA' tipo_comentario,\r\n    replace(replace(replace(replace(observation,chr(10), ''), chr(13), ''),chr(9),''),'|','') comentario\r\nFROM flex.pe_observ_adj_cons\r\nWHERE product_id = :p_servicio_suscrito --{Argumento 2 - servicio_suscrito}\r\n  AND consump_period = :p_id_periodo_consumo --{Argumento 3 - id_periodo_consumo}\r\n  AND consump_type = :p_tipo_consumo --{Argumento 4 - tipo_consumo}\r\n  AND adjustment_date BETWEEN to_date(:p_fecha_creacion, 'YYYY-MM-DD HH24:MI:SS')\r\n                          AND nvl(to_date(:p_fecha_legalizacion, 'YYYY-MM-DD HH24:MI:SS'), sysdate)\r\n",
  "QUERY_CUENTAS_COBRO": "\r\n--datos_cuentas_cobro\r\nWITH data_base as (\r\n    SELECT *\r\n    FROM (\r\n        SELECT /*+ leading(cc, f)*/\r\n            cuconuse ss,\r\n            cucocodi,\r\n            cucoesta,\r\n            cucofact,\r\n            cucovato,\r\n            cucovare,\r\n            cucosacu,\r\n            cucofepa,\r\n            cucovaab,\r\n            cucofeve,\r\n            pefacodi, pefacicl,\r\n            pefaano, pefames,\r\n            pecscons, pecsfeci, pecsfecf\r\n        FROM cuencobr cc,\r\n             factura f,\r\n             perifact pf,\r\n             pericose pc\r\n        WHERE factcodi = cucofact\r\n          AND (\r\n               (factcons = 66)\r\n               OR\r\n               (factprog in (5,97) AND cucoesta = 'P')\r\n          )\r\n          AND pefacodi = factpefa\r\n          AND pefacicl = pecscico\r\n          AND pecscons = pefapecs\r\n          AND cuconuse = :p_servicio_suscrito --{Argumento 1 -servicio suscrito}\r\n        ORDER BY pefaffmo desc\r\n    )\r\n    WHERE rownum <= 8\r\n)\r\nSELECT\r\n    ss servicio_suscrito,\r\n    cucocodi id_cuenta_cobro,\r\n    pefacodi id_periodo_facturacion,\r\n    pefaano anio_facturacion,\r\n    pefames mes_facturacion,\r\n    cucofepa fecha_pago,\r\n    cucovato valor_total,\r\n    cucovaab valor_abonado,\r\n    cucovare valor_reclamo,\r\n    cucosacu valor_pendiente,\r\n    to_char(cucofeve, 'YYYY-MM-DD') fecha_vencimiento,\r\n    (SELECT sum(decode(cargsign, 'DB', cargvalo, 'CR', -cargvalo)) valor\r\n     FROM cargos\r\n     WHERE cargnuse = ss\r\n       AND cargpefa = pefacodi\r\n       AND cargcuco = cucocodi\r\n       AND nvl(cargpeco, pecscons) = pecscons) valor_periodo,\r\n    (SELECT sum(decode(cargsign, 'DB', cargvalo, 'CR', -cargvalo)) valor\r\n     FROM cargos\r\n     WHERE cargnuse = ss\r\n       AND cargpefa = pefacodi\r\n       AND cargcuco = cucocodi\r\n       AND nvl(cargpeco, pecscons) <> pecscons) valor_recuperado\r\nFROM data_base\r\nORDER BY cucofeve desc\r\n",
  "QUERY_DETALLE_CARGOS": "\r\n--datos_detalle_cargos\r\nSELECT\r\n    cargnuse servicio_suscrito,\r\n    cargcuco id_cuenta_cobro,\r\n    (select factpefa from factura,cuencobr where cucocodi = cargcuco and factcodi = cucofact) id_periodo_facturacion,\r\n    cargpeco id_periodo_consumo,\r\n    (select conccodi||'-'||concdesc from concepto where conccodi = cargconc) Concepto,\r\n    (select case(cargcaca)\r\n           when -1 then\r\n                '-1'\r\n           else\r\n                cacacodi||'-'||cacadesc\r\n           end case\r\n     from causcarg\r\n     where cacacodi = cargcaca) causal,\r\n    cargsign signo,\r\n    (select pefaano||'-'||lpad(pefames, 2, '0')\r\n     from perifact\r\n     where pefapecs = cargpeco) periodo_consumo,\r\n    cargdoso documento_soporte,\r\n    to_char(cargfecr, 'YYYY-MM-DD HH24:MI:SS') fecha_creacion_cargo,\r\n    (select proccons||'-'||procdesc from procesos where proccons = cargprog) programa,\r\n    cargtaco id_tarifa,\r\n    nvl(cargunid,1) unidades,\r\n    cargvalo valor\r\nFROM cargos c\r\nWHERE cargcuco = :p_id_cuenta_cobro --{Argumento 1 -cuenta de cobro}\r\nORDER BY cargfecr desc, cargconc\r\n",
  "QUERY_DIM_ESTADO_CORTE_FACTURABLE": "\r\nSELECT escocodi, escodesc, coecfact, coecserv, servdesc\r\nFROM estacort, confesco, servicio\r\nWHERE escocodi = coeccodi\r\n  AND coecserv = servcodi\r\n",
  "QUERY_DETALLE_SOLICITUDES": "\r\nSELECT /*+ leading (m)\r\n        use_nl(a b) use_nl(a c) use_nl(a e)\r\n        use_nl(a f) use_nl(a g) use_nl(a r)\r\n        index (a PK_MO_PACKAGES)\r\n        index (f PK_GE_PERSON)\r\n        index (g PK_GE_ORGANIZAT_AREA)\r\n        index (r PK_GE_SUBSCRIBER) */\r\n        a.package_id package_id,\r\n        r.subscriber_name||' '||r.subs_last_name||' '||r.subs_second_last_name subscriber,\r\n        a.package_type_id||' - '||b.description package_type,\r\n        a.request_date request_date,\r\n        a.motive_status_id||' - '||c.description package_status,\r\n        a.attention_date attention_date,\r\n        a.comment_ comment_,\r\n        a.reception_type_id||' - '||e.description reception_type,\r\n        a.person_id||' - '||f.name_ vendor,\r\n        a.organizat_area_id||' - '||g.name_ organizat_area_id\r\nFROM    mo_packages a, ps_package_type b, ps_motive_status c,\r\n        ge_reception_type e, ge_person f, ge_organizat_area g,\r\n        ge_subscriber r, mo_motive mm\r\nWHERE   mm.product_id = :p_servicio_suscrito\r\n    and a.package_id = mm.package_id\r\n    AND a.package_type_id = b.package_type_id\r\n    AND a.motive_status_id = c.motive_status_id\r\n    AND a.reception_type_id = e.reception_type_id (+)\r\n    AND a.person_id = f.person_id (+)\r\n    AND a.organizat_area_id = g.organizat_area_id (+)\r\n    AND a.subscriber_id = r.subscriber_id (+)\r\norder by 1 desc\r\n",
  "QUERY_SERVICIOS_CONTRATO": "\r\n--servicios_contrato (espejo de datos_basicos_producto, por contrato)\r\nSELECT\r\n    sesunuse servicio_suscrito,\r\n    sesususc contrato,\r\n    p.address_id instalacion,\r\n    (SELECT servcodi||'-'||servdesc FROM servicio WHERE servcodi = sesuserv) servicio,\r\n    to_char(sesufein, 'YYYY-MM-DD') fecha_instalacion,\r\n    to_char(sesufere, 'YYYY-MM-DD') fecha_retiro,\r\n    (select * from (select periodicity from pe_per_his_prod where product_id= sesunuse order by created_date desc) where rownum <= 1) periodicidad,\r\n    (select escocodi||'-'||escodesc from estacort where escocodi = sesuesco) Estado_Corte,\r\n    (select catecodi||'-'||catedesc from categori where catecodi = sesucate) Categoria,\r\n    (select sucacodi||'-'||sucadesc from subcateg where sucacate = sesucate and sucacodi = sesusuca) subcategoria,\r\n    sesucicl as ciclo,\r\n    (select plsucodi||'-'||plsudesc from plansusc where plsucodi = sesuplfa) Plan_Facturacion,\r\n    (select plsucodi||'-'||plsudesc from pr_product,plansusc where commercial_plan_id = plsucodi and product_id = sesunuse) plan_facturacion_pr_product,\r\n    subscriber_name||' '||SUBS_LAST_NAME Nombre_Cliente,\r\n    cl.IDENTIFICATION identificacion,\r\n    (select geograp_location_id||'-'||description from ge_geogra_location g where g.geograp_location_id = d.geograp_location_id) Localidad,\r\n    d.address_parsed direccion,\r\n    cadastral_id PAGINA,\r\n    (select sum(cucosacu) from cuencobr where cuconuse = sesunuse and cucosacu > 0) SALDO_PENDIENTE,\r\n    (select count(1) from cuencobr where cuconuse = sesunuse and cucofeve < sysdate and cucosacu > 0) CUENTAS_VENCIDAS,\r\n    (select sum(cucosacu) from cuencobr where cuconuse = sesunuse and cucofeve < sysdate and cucosacu > 0) SALDO_VENCIDO\r\nFROM servsusc s, pr_product p, ab_address d, suscripc c, ge_subscriber cl\r\nWHERE 1=1\r\nAND sesunuse = product_id\r\nAND c.susccodi = s.sesususc\r\nAND p.product_id = s.sesunuse\r\nAND p.address_id = d.address_id\r\nAND cl.subscriber_id = c.suscclie\r\nAND s.sesususc = :p_contrato -- roster del contrato (incluye retirados; vigencia en Silver)\r\n",
  "QUERY_CONSUMOS_CONTRATO": "\r\nSELECT\r\n    cosssesu servicio_suscrito,\r\n    cosspefa id_periodo_facturacion,\r\n    cosspecs id_periodo_consumo,\r\n    cosscoca consumo,\r\n    (SELECT mecccodi||'-'||meccdesc FROM mecacons WHERE mecccodi = cossmecc) metodo_calculo,\r\n    (SELECT tconcodi||'-'||tcondesc FROM tipocons t WHERE t.tconcodi = cosstcon) tipo_consumo,\r\n    (SELECT cavccodi||'-'||cavcdesc FROM calivaco WHERE cavccodi = cosscavc) calificacion,\r\n    to_char(cossfere, 'YYYY-MM-DD') fecha_registro\r\nFROM conssesu\r\nWHERE cosssesu = :p_servicio_suscrito\r\n  AND cossfere >= ADD_MONTHS(SYSDATE, -6)\r\nORDER BY cosspefa DESC\r\n",
  "QUERY_INVESTIGACION_CONSUMO": "\r\nSELECT\r\n    i.product_id servicio_suscrito,\r\n    (SELECT tconcodi||'-'||tcondesc FROM tipocons t WHERE t.tconcodi = i.consumption_type) tipo_consumo,\r\n    i.consumption_period id_periodo_consumo,\r\n    i.investigate_request solicitud_investigacion,\r\n    i.invest_cons_state_id estado_investigacion,\r\n    (SELECT s.invest_cons_state_id||'-'||s.description FROM pe_invest_cons_state s\r\n      WHERE s.invest_cons_state_id = i.invest_cons_state_id) estado_investigacion_desc,\r\n    to_char(i.register_date, 'YYYY-MM-DD HH24:MI:SS') fecha_registro\r\nFROM pe_invest_consum i\r\nWHERE i.product_id = :p_servicio_suscrito\r\n  AND i.register_date >= ADD_MONTHS(SYSDATE, -6)\r\nORDER BY i.register_date DESC\r\n"
}

print(f"Queries embebidas: {len(QUERIES_ORACLE)}")
for nombre in QUERIES_ORACLE:
    print("  ·", nombre)


In [ ]:
MODELO_ORIGEN = {
    "midas_dim_estado_corte_facturable_bronze": {
        "query_key": "QUERY_DIM_ESTADO_CORTE_FACTURABLE",
        "origen": "ESTACORT + CONFESCO + SERVICIO",
        "depende_de": "ninguna; catálogo completo",
    },
    "midas_ordenes_calidad_pendientes_bronze": {
        "query_key": "QUERY_ORDENES_PENDIENTES",
        "origen": "OR_ORDER_ACTIVITY + OR_ORDER",
        "depende_de": "raíz",
    },
    "midas_datos_basicos_producto_bronze": {
        "query_key": "QUERY_DATOS_BASICOS",
        "origen": "SERVSUSC + PR_PRODUCT + AB_ADDRESS + SUSCRIPC + GE_SUBSCRIBER",
        "depende_de": "instalacion",
    },
    "midas_datos_lecturas_producto_bronze": {
        "query_key": "QUERY_DATOS_LECTURA",
        "origen": "LECTELME + CONSSESU + ELEMMEDI + catálogos",
        "depende_de": "servicio_suscrito",
    },
    "midas_datos_consumos_producto_bronze": {
        "query_key": "QUERY_DATOS_CONSUMOS",
        "origen": "CONSSESU + MECACONS + TIPOCONS + CALIVACO",
        "depende_de": "servicio_suscrito + id_periodo_consumo",
    },
    "midas_datos_ordenes_previa_critica_bronze": {
        "query_key": "QUERY_ORDENES_CRITICA_PEVIA",
        "origen": "CM_ORDECRIT + PE_INVEST_CONSUM + OR_ORDER",
        "depende_de": "periodo_facturacion + SS + tipo_consumo",
    },
    "midas_datos_cometarios_ordenes_bronze": {
        "query_key": "QUERY_COMENTARIOS_ORDENES",
        "origen": "OR_ORDER_COMMENT + PE_OBSERV_ADJ_CONS",
        "depende_de": "orden de crítica + SS + periodo + tipo",
    },
    "midas_datos_cuentas_cobro_bronze": {
        "query_key": "QUERY_CUENTAS_COBRO",
        "origen": "CUENCOBR + FACTURA + PERIFACT + PERICOSE",
        "depende_de": "servicio_suscrito",
    },
    "midas_datos_detalle_cargos_bronze": {
        "query_key": "QUERY_DETALLE_CARGOS",
        "origen": "CARGOS + CONCEPTO + CAUSCARG + PROCESOS",
        "depende_de": "id_cuenta_cobro",
    },
    "midas_datos_detalle_solicitudes_bronze": {
        "query_key": "QUERY_DETALLE_SOLICITUDES",
        "origen": "MO_PACKAGES + MO_MOTIVE + catálogos",
        "depende_de": "servicio_suscrito",
    },
    "midas_datos_servicios_contrato_bronze": {
        "query_key": "QUERY_SERVICIOS_CONTRATO",
        "origen": "SERVSUSC y los mismos joins de datos básicos",
        "depende_de": "contrato",
    },
    "midas_datos_consumos_contrato_bronze": {
        "query_key": "QUERY_CONSUMOS_CONTRATO",
        "origen": "CONSSESU + catálogos",
        "depende_de": "cada servicio_suscrito del contrato",
    },
    "midas_datos_investigacion_consumo_bronze": {
        "query_key": "QUERY_INVESTIGACION_CONSUMO",
        "origen": "PE_INVEST_CONSUM + TIPOCONS + PE_INVEST_CONS_STATE",
        "depende_de": "cada servicio_suscrito",
    },
}

display(pd.DataFrame([
    {"tabla_destino": tabla, **meta}
    for tabla, meta in MODELO_ORIGEN.items()
]))


## 0.4 Ejecutor guiado Oracle → modelo lógico Bronze

La celda siguiente recorre el grafo en el mismo orden conceptual que la cadena:

```
orden → básicos → lecturas → consumos → crítica → comentarios
     ├─→ cuentas → cargos
     ├─→ solicitudes
     └─→ servicios del contrato → consumos hermanos / investigación

dimensión facturable (independiente)
```

Cada resultado se guarda en:

- `RESULTADOS_ORACLE[tabla]`: columnas tal como las devuelve JDBC.
- `RESULTADOS_DESTINO[tabla]`: nombres en minúscula como quedarían en Bronze.
- `EJECUCIONES_ORACLE`: query, parámetros, cantidad de filas y estado.

La ejecución es tolerante a vacíos: una tabla sin datos no impide explorar las demás ramas
que sí tengan anclas disponibles.


In [ ]:
RESULTADOS_ORACLE = {}
RESULTADOS_DESTINO = {}
EJECUCIONES_ORACLE = []
ANCLAS = dict(ANCLAS_MANUALES)

MAPEO_SOLICITUDES = {
    "SERVICIO_SUSCRITO": "servicio_suscrito",
    "PACKAGE_ID": "id_solicitud",
    "SUBSCRIBER": "usuario",
    "PACKAGE_TYPE": "tipo_solicitud",
    "REQUEST_DATE": "fecha_solicitud",
    "PACKAGE_STATUS": "estado_solicitud",
    "ATTENTION_DATE": "fecha_atencion_solicitud",
    "COMMENT_": "comentario",
    "RECEPTION_TYPE": "medio_recepcion",
    "VENDOR": "analista",
    "ORGANIZAT_AREA_ID": "area_organizacional",
}

def _columna(df, nombre):
    if df is None:
        return None
    return next((c for c in df.columns if c.lower() == nombre.lower()), None)

def _valor(df, nombre, fila=0):
    columna = _columna(df, nombre)
    if columna is None or df is None or len(df) <= fila:
        return None
    valor = df.iloc[fila][columna]
    return None if pd.isna(valor) else valor

def _a_entero(valor):
    try:
        return int(valor)
    except Exception:
        return None

def _normalizar_destino(tabla, df, params):
    if df is None:
        return pd.DataFrame()
    salida = df.copy()
    if tabla == "midas_datos_detalle_solicitudes_bronze":
        if "SERVICIO_SUSCRITO" not in salida.columns:
            salida.insert(0, "SERVICIO_SUSCRITO", params.get("p_servicio_suscrito"))
        presentes = [c for c in MAPEO_SOLICITUDES if c in salida.columns]
        salida = salida[presentes].rename(columns=MAPEO_SOLICITUDES)
        return salida
    salida.columns = [str(c).lower() for c in salida.columns]
    return salida

def ejecutar_tabla(tabla, params=None, sql_personalizado=None, limite=MAX_FILAS_ORACLE):
    meta = MODELO_ORIGEN[tabla]
    query_key = meta["query_key"]
    parametros = dict(params or {})
    parametros["__limite_midas"] = limite
    sql_base = sql_personalizado or QUERIES_ORACLE[query_key]
    print("\n" + "=" * 100)
    print(f"{tabla}  ←  {query_key}")
    print("Origen:", meta["origen"])
    print("Parámetros:", {k: v for k, v in parametros.items() if k != "__limite_midas"})
    try:
        df = ejecutar_select(limitar_sql(sql_base, limite), parametros)
        RESULTADOS_ORACLE[tabla] = df
        RESULTADOS_DESTINO[tabla] = _normalizar_destino(tabla, df, parametros)
        EJECUCIONES_ORACLE.append({
            "tabla_destino": tabla,
            "query_key": query_key,
            "estado": "OK",
            "filas": len(df),
            "parametros": str({k: v for k, v in parametros.items() if k != "__limite_midas"}),
        })
        mostrar_pd(df, f"Salida Oracle ({len(df)} filas)")
        print("Columnas destino:", list(RESULTADOS_DESTINO[tabla].columns))
        return df
    except Exception as error:
        RESULTADOS_ORACLE[tabla] = pd.DataFrame()
        RESULTADOS_DESTINO[tabla] = pd.DataFrame()
        EJECUCIONES_ORACLE.append({
            "tabla_destino": tabla,
            "query_key": query_key,
            "estado": "ERROR",
            "filas": 0,
            "parametros": str({k: v for k, v in parametros.items() if k != "__limite_midas"}),
            "error": str(error),
        })
        print(f"❌ {type(error).__name__}: {error}")
        return pd.DataFrame()

def guardar_resultado_compuesto(tabla, frames, params_resumen):
    frames_validos = [df for df in frames if df is not None and not df.empty]
    combinado = pd.concat(frames_validos, ignore_index=True) if frames_validos else pd.DataFrame()
    RESULTADOS_ORACLE[tabla] = combinado
    RESULTADOS_DESTINO[tabla] = _normalizar_destino(tabla, combinado, {})
    EJECUCIONES_ORACLE.append({
        "tabla_destino": tabla,
        "query_key": MODELO_ORIGEN[tabla]["query_key"],
        "estado": "OK" if frames_validos else "SIN_FILAS",
        "filas": len(combinado),
        "parametros": params_resumen,
    })
    mostrar_pd(combinado, f"Resultado consolidado de {tabla}")
    return combinado


In [ ]:
# 1) Dimensión independiente.
df_dim_origen = ejecutar_tabla("midas_dim_estado_corte_facturable_bronze")

# 2) Raíz: orden específica o muestra general priorizando la actividad configurada.
sql_ordenes = QUERIES_ORACLE["QUERY_ORDENES_PENDIENTES"]
params_ordenes = {}
if ANCLAS.get("id_orden") is not None:
    sql_ordenes = f"SELECT * FROM ({sql_ordenes.strip()}) q_orden WHERE q_orden.id_orden = :p_id_orden_filtro"
    params_ordenes["p_id_orden_filtro"] = ANCLAS["id_orden"]
elif ANCLAS.get("servicio_suscrito") is not None:
    sql_ordenes = f"SELECT * FROM ({sql_ordenes.strip()}) q_orden WHERE q_orden.servicio_suscrito = :p_ss_filtro"
    params_ordenes["p_ss_filtro"] = ANCLAS["servicio_suscrito"]
elif ANCLAS.get("contrato") is not None:
    sql_ordenes = f"SELECT * FROM ({sql_ordenes.strip()}) q_orden WHERE q_orden.contrato = :p_contrato_filtro"
    params_ordenes["p_contrato_filtro"] = ANCLAS["contrato"]
elif ANCLAS.get("instalacion") is not None:
    sql_ordenes = f"SELECT * FROM ({sql_ordenes.strip()}) q_orden WHERE q_orden.instalacion = :p_instalacion_filtro"
    params_ordenes["p_instalacion_filtro"] = ANCLAS["instalacion"]

df_ordenes_origen = ejecutar_tabla(
    "midas_ordenes_calidad_pendientes_bronze",
    params=params_ordenes,
    sql_personalizado=sql_ordenes,
    limite=max(MAX_FILAS_ORACLE, 100),
)

if not df_ordenes_origen.empty:
    col_actividad = _columna(df_ordenes_origen, "actividad")
    candidatos = df_ordenes_origen
    if col_actividad and ACTIVIDAD_PREFERIDA:
        preferidas = df_ordenes_origen[
            df_ordenes_origen[col_actividad].astype(str).str.startswith(ACTIVIDAD_PREFERIDA)
        ]
        if not preferidas.empty:
            candidatos = preferidas
    fila = candidatos.iloc[0]
    for clave in ("id_orden", "servicio_suscrito", "contrato", "instalacion"):
        if ANCLAS.get(clave) is None:
            columna = _columna(candidatos, clave)
            if columna:
                ANCLAS[clave] = _a_entero(fila[columna])

# Si solo se conoce el contrato y no hay orden pendiente, el roster permite continuar.
df_servicios_contrato_origen = pd.DataFrame()
if ANCLAS.get("contrato") is not None and (
    ANCLAS.get("servicio_suscrito") is None or ANCLAS.get("instalacion") is None
):
    df_servicios_contrato_origen = ejecutar_tabla(
        "midas_datos_servicios_contrato_bronze",
        {"p_contrato": ANCLAS["contrato"]},
        limite=max(MAX_FILAS_ORACLE, MAX_SERVICIOS_CONTRATO),
    )
    if not df_servicios_contrato_origen.empty:
        candidatos_ss = df_servicios_contrato_origen
        col_retiro = _columna(candidatos_ss, "fecha_retiro")
        if col_retiro:
            vigentes = candidatos_ss[
                candidatos_ss[col_retiro].isna()
                | (candidatos_ss[col_retiro].astype(str).str.strip() == "")
            ]
            if not vigentes.empty:
                candidatos_ss = vigentes
        if ANCLAS.get("servicio_suscrito") is None:
            ANCLAS["servicio_suscrito"] = _a_entero(_valor(candidatos_ss, "servicio_suscrito"))
        if ANCLAS.get("instalacion") is None:
            ANCLAS["instalacion"] = _a_entero(_valor(candidatos_ss, "instalacion"))

print("\nANCLAS DESPUÉS DE LA RAÍZ:", ANCLAS)


In [ ]:
# 3) Ficha básica por instalación.
if ANCLAS.get("instalacion") is not None:
    df_basicos_origen = ejecutar_tabla(
        "midas_datos_basicos_producto_bronze",
        {"address_id": ANCLAS["instalacion"]},
    )
else:
    df_basicos_origen = pd.DataFrame()
    print("⚪ Datos básicos omitidos: falta instalacion.")

# 4) Lecturas por SS.
if ANCLAS.get("servicio_suscrito") is not None:
    df_lecturas_origen = ejecutar_tabla(
        "midas_datos_lecturas_producto_bronze",
        {"servicio_suscrito": ANCLAS["servicio_suscrito"]},
    )
    if not df_lecturas_origen.empty:
        if ANCLAS.get("id_periodo_consumo") is None:
            ANCLAS["id_periodo_consumo"] = _a_entero(_valor(df_lecturas_origen, "id_periodo_consumo"))
        if ANCLAS.get("id_periodo_facturacion") is None:
            ANCLAS["id_periodo_facturacion"] = _a_entero(_valor(df_lecturas_origen, "id_periodo_facturacion"))
        if ANCLAS.get("tipo_consumo") is None:
            ANCLAS["tipo_consumo"] = _a_entero(_valor(df_lecturas_origen, "tipocons"))
else:
    df_lecturas_origen = pd.DataFrame()
    print("⚪ Lecturas omitidas: falta servicio_suscrito.")

print("\nANCLAS DESPUÉS DE LECTURAS:", ANCLAS)


In [ ]:
# 5) Detalle de consumos del periodo seleccionado.
if ANCLAS.get("servicio_suscrito") is not None and ANCLAS.get("id_periodo_consumo") is not None:
    df_consumos_origen = ejecutar_tabla(
        "midas_datos_consumos_producto_bronze",
        {
            "p_servicio_suscrito": ANCLAS["servicio_suscrito"],
            "p_id_periodo_consumo": ANCLAS["id_periodo_consumo"],
        },
    )
else:
    df_consumos_origen = pd.DataFrame()
    print("⚪ Consumos omitidos: faltan SS o periodo de consumo.")

# 6) Órdenes de crítica/previa.
requisitos_critica = (
    ANCLAS.get("servicio_suscrito") is not None
    and ANCLAS.get("id_periodo_facturacion") is not None
)
if requisitos_critica:
    df_criticas_origen = ejecutar_tabla(
        "midas_datos_ordenes_previa_critica_bronze",
        {
            "p_id_periodo_facturacion": ANCLAS["id_periodo_facturacion"],
            "p_servicio_suscrito": ANCLAS["servicio_suscrito"],
            "p_tipo_consumo": ANCLAS.get("tipo_consumo"),
        },
    )
else:
    df_criticas_origen = pd.DataFrame()
    print("⚪ Crítica omitida: faltan SS o periodo de facturación.")


In [ ]:
# 7) Comentarios de una orden de crítica/previa.
if not df_criticas_origen.empty:
    id_orden_critica = _a_entero(_valor(df_criticas_origen, "id_orden"))
    fecha_creacion = _valor(df_criticas_origen, "fecha_creacion_orden")
    fecha_legalizacion = _valor(df_criticas_origen, "fecha_legalizacion_orden")
    params_comentarios = {
        "p_id_orden": id_orden_critica,
        "p_servicio_suscrito": ANCLAS["servicio_suscrito"],
        "p_id_periodo_consumo": ANCLAS["id_periodo_consumo"],
        "p_tipo_consumo": ANCLAS.get("tipo_consumo"),
        "p_fecha_creacion": fecha_creacion,
        "p_fecha_legalizacion": fecha_legalizacion,
    }
    df_comentarios_origen = ejecutar_tabla(
        "midas_datos_cometarios_ordenes_bronze",
        params_comentarios,
    )
else:
    df_comentarios_origen = pd.DataFrame()
    print("⚪ Comentarios omitidos: la query de crítica no devolvió una orden.")


In [ ]:
# 8) Cuentas de cobro y detalle de cargos.
if ANCLAS.get("servicio_suscrito") is not None:
    df_cuentas_origen = ejecutar_tabla(
        "midas_datos_cuentas_cobro_bronze",
        {"p_servicio_suscrito": ANCLAS["servicio_suscrito"]},
    )
else:
    df_cuentas_origen = pd.DataFrame()
    print("⚪ Cuentas omitidas: falta SS.")

if not df_cuentas_origen.empty:
    id_cuenta = _a_entero(_valor(df_cuentas_origen, "id_cuenta_cobro"))
    df_cargos_origen = ejecutar_tabla(
        "midas_datos_detalle_cargos_bronze",
        {"p_id_cuenta_cobro": id_cuenta},
    )
else:
    df_cargos_origen = pd.DataFrame()
    print("⚪ Cargos omitidos: cuentas no devolvió id_cuenta_cobro.")


In [ ]:
# 9) Solicitudes del SS.
if ANCLAS.get("servicio_suscrito") is not None:
    df_solicitudes_origen = ejecutar_tabla(
        "midas_datos_detalle_solicitudes_bronze",
        {"p_servicio_suscrito": ANCLAS["servicio_suscrito"]},
    )
else:
    df_solicitudes_origen = pd.DataFrame()
    print("⚪ Solicitudes omitidas: falta SS.")

# 10) Roster completo del contrato (se reutiliza si ya resolvió anclas).
if not df_servicios_contrato_origen.empty:
    print("Se reutiliza el roster del contrato ejecutado durante la resolución de anclas.")
elif ANCLAS.get("contrato") is not None:
    df_servicios_contrato_origen = ejecutar_tabla(
        "midas_datos_servicios_contrato_bronze",
        {"p_contrato": ANCLAS["contrato"]},
        limite=max(MAX_FILAS_ORACLE, MAX_SERVICIOS_CONTRATO),
    )
else:
    df_servicios_contrato_origen = pd.DataFrame()
    print("⚪ Servicios del contrato omitidos: falta contrato.")


In [ ]:
# 11 y 12) Historial e investigación para los SS hermanos.
ss_contrato = []
if not df_servicios_contrato_origen.empty:
    col_ss = _columna(df_servicios_contrato_origen, "servicio_suscrito")
    if col_ss:
        ss_contrato = [
            _a_entero(v) for v in df_servicios_contrato_origen[col_ss].dropna().unique()
        ]
        ss_contrato = [v for v in ss_contrato if v is not None][:MAX_SERVICIOS_CONTRATO]
if not ss_contrato and ANCLAS.get("servicio_suscrito") is not None:
    ss_contrato = [ANCLAS["servicio_suscrito"]]

frames_consumos_contrato = []
frames_investigacion = []
for ss in ss_contrato:
    print(f"\n--- SS hermano {ss} ---")
    try:
        df = ejecutar_select(
            limitar_sql(QUERIES_ORACLE["QUERY_CONSUMOS_CONTRATO"]),
            {"p_servicio_suscrito": ss, "__limite_midas": MAX_FILAS_ORACLE},
        )
        frames_consumos_contrato.append(df)
    except Exception as error:
        print(f"  ❌ consumos_contrato: {error}")
    try:
        df = ejecutar_select(
            limitar_sql(QUERIES_ORACLE["QUERY_INVESTIGACION_CONSUMO"]),
            {"p_servicio_suscrito": ss, "__limite_midas": MAX_FILAS_ORACLE},
        )
        frames_investigacion.append(df)
    except Exception as error:
        print(f"  ❌ investigacion_consumo: {error}")

df_consumos_contrato_origen = guardar_resultado_compuesto(
    "midas_datos_consumos_contrato_bronze",
    frames_consumos_contrato,
    f"servicios_suscritos={ss_contrato}",
)
df_investigacion_origen = guardar_resultado_compuesto(
    "midas_datos_investigacion_consumo_bronze",
    frames_investigacion,
    f"servicios_suscritos={ss_contrato}",
)


## 0.5 Resumen, inspección libre y cierre

El resumen permite distinguir:

- `OK`: la query se ejecutó, aunque legítimamente puede devolver cero filas.
- `ERROR`: falló la consulta, un bind, un permiso o una tabla de Oracle.
- `SIN_FILAS`: las consultas iteradas de A3/A4 no encontraron datos para la muestra.

Para volver a visualizar una tabla:

```python
mostrar_pd(RESULTADOS_ORACLE["midas_datos_lecturas_producto_bronze"])
mostrar_pd(RESULTADOS_DESTINO["midas_datos_lecturas_producto_bronze"])
```

La conexión se cierra al final. Si luego quieres ejecutar otra consulta, llama nuevamente
a `conectar_oracle()`; el helper la reabre de forma segura.


In [ ]:
resumen_oracle = pd.DataFrame(EJECUCIONES_ORACLE)
mostrar_pd(resumen_oracle, "RESUMEN DE EJECUCIÓN DESDE ORACLE")

print("\nAnclas efectivas:", ANCLAS)
print("\nResultados disponibles:")
for tabla, df in RESULTADOS_DESTINO.items():
    print(f"  · {tabla}: {len(df)} filas · {len(df.columns)} columnas")

cerrar_oracle()


---
# Exploración opcional de las tablas Bronze

Desde aquí continúa el explorador original de Unity Catalog. Como la versión del repo aún
no está cargada en Databricks, es normal que algunas o todas las tablas no existan.

Esta parte no es necesaria para explorar Oracle: los resultados directos ya quedaron en
`RESULTADOS_ORACLE` y `RESULTADOS_DESTINO`. Cuando las Bronze se desplieguen, esta segunda
ruta permitirá comparar origen y destino en el mismo cuaderno.


---
# 1. Configuración

Lo primero es decirle al cuaderno **dónde** están las tablas.

En Databricks, una tabla se identifica con tres niveles: `catalogo.esquema.tabla`.
En MIDAS el **catálogo cambia según el ambiente** (`dllo` desarrollo, `uat` pruebas,
`pdn` producción) pero **los nombres de tabla son siempre los mismos**.

Los *widgets* de abajo crean cajitas de texto en la parte superior del notebook para que
puedas cambiar el ambiente sin editar código.

In [ ]:
# Widgets: aparecen como cajas de texto arriba del notebook.
# Si ya existen de una ejecución anterior, dbutils no los duplica.
dbutils.widgets.text("catalogo", "dllo", "1. Catálogo (dllo / uat / pdn)")
dbutils.widgets.text("esquema",  "facturacion", "2. Esquema")

CATALOGO = dbutils.widgets.get("catalogo").strip()
ESQUEMA  = dbutils.widgets.get("esquema").strip()
PREFIJO  = f"{CATALOGO}.{ESQUEMA}"

print(f"Trabajando contra: {PREFIJO}")
print("\nSi el nombre del esquema no es 'facturacion', cámbialo en el widget de arriba")
print("y vuelve a ejecutar esta celda.")

### 1.1 Funciones auxiliares

Estas tres funciones se usan en todo el cuaderno. Vale la pena entenderlas una vez:

| Función | Qué hace |
|---|---|
| `existe(tabla)` | Dice si la tabla existe. Evita que el notebook se caiga si algo aún no está desplegado |
| `explorar(tabla)` | Muestra el esquema, el conteo de filas y una muestra de datos |
| `perfil(tabla, columna)` | Muestra los valores más frecuentes de una columna — ideal para entender catálogos |

In [ ]:
from pyspark.sql import functions as F

def existe(tabla: str) -> bool:
    """Devuelve True si la tabla existe en el catálogo configurado."""
    try:
        spark.table(f"{PREFIJO}.{tabla}")
        return True
    except Exception:
        return False


def explorar(tabla: str, n: int = 5, solo_columnas: list = None):
    """
    Muestra la 'ficha técnica' de una tabla Bronze:
      1) si existe,  2) cuántas filas tiene,  3) qué columnas,  4) una muestra.
    """
    completo = f"{PREFIJO}.{tabla}"
    if not existe(tabla):
        print(f"⚠️  {tabla} NO existe todavía en {PREFIJO}.")
        print("    Puede ser normal si aún no se ha desplegado esa carga.")
        return None

    df = spark.table(completo)
    filas = df.count()
    print(f"✅ {tabla}")
    print(f"   Filas: {filas:,}   |   Columnas: {len(df.columns)}")
    print(f"   Columnas: {', '.join(df.columns)}\n")

    muestra = df.select(*solo_columnas) if solo_columnas else df
    display(muestra.limit(n))
    return df


def perfil(tabla: str, columna: str, top: int = 15):
    """
    Muestra los valores más frecuentes de una columna.
    Sirve para descubrir catálogos: '¿qué valores puede tomar estado_corte?'
    """
    if not existe(tabla):
        print(f"⚠️  {tabla} no existe.")
        return
    df = spark.table(f"{PREFIJO}.{tabla}")
    if columna not in df.columns:
        print(f"⚠️  La columna '{columna}' no está en {tabla}.")
        print(f"    Columnas disponibles: {', '.join(df.columns)}")
        return
    print(f"Valores más frecuentes de {tabla}.{columna}:\n")
    display(
        df.groupBy(columna)
          .agg(F.count("*").alias("frecuencia"))
          .orderBy(F.desc("frecuencia"))
          .limit(top)
    )

print("Funciones listas: existe() · explorar() · perfil()")

---
# 2. El modelo mental del negocio

Antes de mirar una sola tabla, hay que entender **cómo piensa EPM su negocio**. Todo el
modelo de datos es un reflejo de esta jerarquía, y si la entiendes, el 80% de las
consultas se vuelven obvias.

```
┌──────────────────────────────────────────────────────────┐
│  CLIENTE / SUSCRIPTOR                                    │
│  "Quién es la persona o empresa"                         │
│  Oracle: ge_subscriber.SUBSCRIBER_ID                     │
└───────────────────────┬──────────────────────────────────┘
                        │ 1 cliente puede tener N contratos
┌───────────────────────▼──────────────────────────────────┐
│  CONTRATO  (también llamado "cuenta" o "suscripción")     │
│  "La cuenta que agrupa los servicios de un predio"       │
│  Oracle: servsusc.SESUSUSC  ·  suscripc.SUSCCODI         │
│                                                           │
│  ★ EL ANALISTA BUSCA A ESTE NIVEL, no por servicio.       │
└───────────────────────┬──────────────────────────────────┘
                        │ ~4.6 servicios por contrato (medido)
┌───────────────────────▼──────────────────────────────────┐
│  SERVICIO SUSCRITO (SS)   ★★★  LA ENTIDAD CENTRAL         │
│  "El agua de ese predio", "la energía de ese predio"     │
│  Oracle: servsusc.SESUNUSE  =  pr_product.PRODUCT_ID     │
│                                                           │
│  Casi TODAS las tablas cuelgan de aquí.                  │
└───────────────────────┬──────────────────────────────────┘
                        │ 1 servicio tiene N periodos
┌───────────────────────▼──────────────────────────────────┐
│  PERIODO DE CONSUMO                                       │
│  "El mes que se factura"                                  │
│  Oracle: pericose.PECSCONS                                │
└──────────────────────────────────────────────────────────┘
```

### Por qué esto importa tanto para el Caso 2

El **Caso 1** (diferencia acueducto/alcantarillado) compara dos servicios de **la misma
instalación** — se queda en el nivel del predio.

El **Caso 2** (variación significativa) razona a nivel de **contrato**. Un ejemplo real
de las sesiones grabadas con el analista:

> La orden es de **energía**. El analista no encuentra la explicación ahí, así que abre
> el **agua del mismo contrato** y encuentra una revisión que dice *"aumento de uso, ocupó
> el local hace 2 meses"*. Cierra la orden de energía como *recién ocupado*, con evidencia
> que vino de otro servicio.

**Sin traer todos los servicios del contrato, esa orden es irresoluble.** Esa es
exactamente la razón de existir de las tablas nuevas `servicios_contrato` y
`consumos_contrato`.

### Las tres llaves que encadenan todo

| Llave | Qué representa | Cómo se llama en Bronze |
|---|---|---|
| `instalacion` | El predio físico | `instalacion` |
| `servicio_suscrito` | El servicio de ese predio (agua, gas, energía…) | `servicio_suscrito` |
| `contrato` | La cuenta que agrupa los servicios | `contrato` |

---
# 3. Cómo leer los nombres de FLEX (la regla del 4+4)

Cuando abras las queries de Oracle vas a ver columnas como `SESUNUSE`, `COSSCOCA` o
`LEEMOBLE`. Parecen aleatorias. **No lo son.**

FLEX (el sistema comercial de EPM, heredero de *Open Smartflex*) usa un esquema de
**4 letras + 4 letras**:

```
        SESUNUSE
        └─┬┘└─┬┘
          │   └──── NUSE = NÚmero de SErvicio       (el campo)
          └──────── SESU = SErvicio SUscrito        (la tabla)


        COSSCOCA
        └─┬┘└─┬┘
          │   └──── COCA = COnsumo CAlculado        (el campo)
          └──────── COSS = CONSsesu                 (la tabla)
```

**Si sabes el prefijo, sabes de qué tabla viene la columna:**

| Prefijo | Tabla Oracle | Dominio |
|---|---|---|
| `SESU` | `servsusc` | Servicio suscrito |
| `LEEM` | `lectelme` | Lecturas del elemento de medida |
| `COSS` | `conssesu` | Consumos del servicio suscrito |
| `CUCO` | `cuencobr` | Cuentas de cobro |
| `CARG` | `cargos` | Cargos facturados |
| `ELME` | `elemmedi` | Elemento de medición (el medidor) |
| `PECS` | `pericose` | Periodo de consumo |
| `PEFA` | `perifact` | Periodo de facturación |

Las tablas **modernas** de FLEX usan inglés con guion bajo: `OR_` órdenes, `PR_` producto,
`GE_` catálogos generales, `MO_` solicitudes, `AB_` direcciones, `PE_` procesos especiales,
`CM_` crítica.

### El patrón "código-descripción" ★

Casi todo atributo categórico se guarda como un **código numérico** que apunta a una
**tabla catálogo**. Las queries de MIDAS los concatenan al extraer:

```sql
(SELECT escocodi || '-' || escodesc FROM estacort WHERE escocodi = sesuesco) AS estado_corte
```

Por eso en Bronze **no ves un `4` suelto**, ves `"4-Orden suspension total"`.

> **Consecuencia práctica muy importante:** para filtrar por código hay que usar
> `startswith()` o `split()`, no igualdad. Por ejemplo, para quedarte con los servicios
> en conexión: `WHERE estado_corte LIKE '1-%'`, no `WHERE estado_corte = 1`.

Esta decisión de diseño (resolver los catálogos *inline*) es la razón por la que el
proyecto **no necesita 9 tablas de dimensión**: los códigos ya llegan traducidos.

In [ ]:
# 🔍 EXPLORA — comprueba tú mismo el patrón código-descripción.
# Fíjate en que los valores NO son números: son "codigo-descripcion".
perfil("midas_datos_basicos_producto_bronze", "estado_corte", top=15)

---
# 4. Inventario: las 13 cargas del Caso 2

La capa Bronze se gobierna con un **plano de control**: dos tablas que dicen qué se carga,
en qué orden y cómo salió.

| Tabla de control | Para qué sirve |
|---|---|
| `midas_control_cargas` | El **catálogo de cargas**: qué tabla, qué query, en qué orden, encadenada o no |
| `midas_log_cargas` | El **diario de ejecución**: cada corrida, cuántas filas, éxito o fallo |

Ambas las comparten varios frameworks, y se distinguen por la columna **`job_name`**
(para MIDAS Bronze es `'midas_bronze'`). Esto es importante: si consultas sin filtrar por
`job_name` verás también las cargas de otros frameworks.

### Las 13 cargas

| Grupo | Tipo | Tablas |
|---|---|---|
| **Cadena Caso 1** (8) | `FULL_CHAINED` | Las 8 originales, **intactas** |
| **Dimensión** (1) | `QUERY_FULL_OVERWRITE` | `midas_dim_estado_corte_facturable_bronze` |
| **Nuevas Caso 2** (4) | `FULL_CHAINED` | `detalle_solicitudes`, `servicios_contrato`, `consumos_contrato`, `investigacion_consumo` |

Más `midas_parametros`, que no es una carga sino una semilla de configuración.

**¿Qué significan los dos tipos de carga?**

- **`FULL_CHAINED`** — *encadenada*. La query necesita un parámetro que sale de una tabla
  anterior. Ejemplo: para traer las lecturas hay que saber primero **de qué servicio
  suscrito**, y eso lo da la tabla raíz de órdenes.
- **`QUERY_FULL_OVERWRITE`** — *autocontenida*. La query se basta sola y se materializa
  completa cada día. La dimensión de estado facturable es así porque no depende de ninguna
  orden: es un catálogo de reglas.

In [ ]:
# El catálogo de cargas: qué se carga y en qué orden.
if existe("midas_control_cargas"):
    display(
        spark.table(f"{PREFIJO}.midas_control_cargas")
             .filter(F.col("job_name") == "midas_bronze")
             .orderBy("orden_ejecucion")
    )
else:
    print("⚠️  midas_control_cargas no existe en este catálogo.")
    print("    Revisa el nombre del esquema en el widget.")

In [ ]:
# El diario de ejecución: cómo salió la última corrida.
# Útil para saber si los datos que estás viendo son de hoy y si alguna carga falló.
if existe("midas_log_cargas"):
    display(
        spark.table(f"{PREFIJO}.midas_log_cargas")
             .filter(F.col("job_name") == "midas_bronze")
             .orderBy(F.desc("fecha_inicio"))
             .limit(30)
    )
else:
    print("⚠️  midas_log_cargas no existe todavía.")

In [ ]:
# Los parámetros: los "códigos mágicos" del negocio, descableados.
# Aquí es donde viven 993 (Caso 2), 1019 (Caso 1), 883 (órdenes de calidad), etc.
if existe("midas_parametros"):
    display(spark.table(f"{PREFIJO}.midas_parametros"))
else:
    print("⚠️  midas_parametros no existe todavía.")

### 1.2 Contrato técnico verificado contra el repo

Esta versión ya no adivina nombres. Se contrastaron tres fuentes:

- `src/midas/db/queries.py`: alias que salen de Oracle.
- `notebooks/00_creacion_objetos_midas.py`: seed vigente de **13 cargas** y nombres destino.
- `src/midas/main_ingestion.py` + `src/midas/ingestion.py`: configuración y creación del destino.

**Hallazgo sobre el “DDL de destino”.** El repo no contiene un DDL estático para las 13
Bronze. En la primera carga, Spark crea cada Delta con el esquema inferido del Parquet
(`saveAsTable`); en cargas posteriores usa `insertInto(..., overwrite=True)`, que es
**posicional**. La excepción relevante es
`midas_datos_detalle_solicitudes_bronze`: ya existía y el código adapta explícitamente
los alias Oracle a sus 11 nombres reales en español.

La siguiente celda contiene el contrato campo a campo derivado de las queries y valida
el ambiente actual. Una columna extra se informa, pero solo una columna faltante marca error.


In [ ]:
# Contrato de columnas reales: alias Oracle -> minúsculas aplicadas por DataIngestor.
ESQUEMAS_REALES = {
    "midas_ordenes_calidad_pendientes_bronze": [
        "id_orden", "servicio_suscrito", "instalacion", "contrato", "fecha_creacion",
        "actividad", "estado_orden", "comentario_orden",
    ],
    "midas_datos_basicos_producto_bronze": [
        "servicio_suscrito", "contrato", "instalacion", "servicio", "fecha_instalacion",
        "fecha_retiro", "periodicidad", "estado_corte", "categoria", "subcategoria", "ciclo",
        "plan_facturacion", "plan_facturacion_pr_product", "nombre_cliente", "identificacion",
        "localidad", "direccion", "pagina", "saldo_pendiente", "cuentas_vencidas", "saldo_vencido",
    ],
    "midas_datos_lecturas_producto_bronze": [
        "servicio_suscrito", "id_periodo_consumo", "id_periodo_facturacion",
        "fecha_ini_consumo", "fecha_fin_consumo", "dias_consumo", "tipo_consumo", "tipocons",
        "medidor", "constante", "digitos_medidor", "lectura_anterior", "lectura_actual",
        "consumo_calculado", "consumo_facturado", "limite_inferior", "limite_superior",
        "observacion_lectura", "observacion_lectura_2", "observacion_lectura_3", "pno",
    ],
    "midas_datos_consumos_producto_bronze": [
        "servicio_suscrito", "id_periodo_consumo", "id_periodo_facturacion", "anio_facturacion",
        "mes_facturacion", "ciclo", "ciclo_operativo", "fecha_registro", "metodo_calculo",
        "tipo_consumo", "consumo", "funcion_calculo", "calificacion",
    ],
    "midas_datos_ordenes_previa_critica_bronze": [
        "id_orden", "servicio_suscrito", "tipo_consumo", "id_periodo_consumo", "tipo_trabajo",
        "actividad", "fecha_creacion_orden", "fecha_legalizacion_orden", "estado", "analista_legaliza",
    ],
    "midas_datos_cometarios_ordenes_bronze": [
        "id_orden", "servicio_suscrito", "fecha_registro", "tipo_comentario", "comentario",
    ],
    "midas_datos_cuentas_cobro_bronze": [
        "servicio_suscrito", "id_cuenta_cobro", "id_periodo_facturacion", "anio_facturacion",
        "mes_facturacion", "fecha_pago", "valor_total", "valor_abonado", "valor_reclamo",
        "valor_pendiente", "fecha_vencimiento", "valor_periodo", "valor_recuperado",
    ],
    "midas_datos_detalle_cargos_bronze": [
        "servicio_suscrito", "id_cuenta_cobro", "id_periodo_facturacion", "id_periodo_consumo",
        "concepto", "causal", "signo", "periodo_consumo", "documento_soporte",
        "fecha_creacion_cargo", "programa", "id_tarifa", "unidades", "valor",
    ],
    "midas_dim_estado_corte_facturable_bronze": [
        "escocodi", "escodesc", "coecfact", "coecserv", "servdesc",
    ],
    "midas_datos_detalle_solicitudes_bronze": [
        "servicio_suscrito", "id_solicitud", "usuario", "tipo_solicitud", "fecha_solicitud",
        "estado_solicitud", "fecha_atencion_solicitud", "comentario", "medio_recepcion",
        "analista", "area_organizacional",
    ],
    "midas_datos_servicios_contrato_bronze": [
        "servicio_suscrito", "contrato", "instalacion", "servicio", "fecha_instalacion",
        "fecha_retiro", "periodicidad", "estado_corte", "categoria", "subcategoria", "ciclo",
        "plan_facturacion", "plan_facturacion_pr_product", "nombre_cliente", "identificacion",
        "localidad", "direccion", "pagina", "saldo_pendiente", "cuentas_vencidas", "saldo_vencido",
    ],
    "midas_datos_consumos_contrato_bronze": [
        "servicio_suscrito", "id_periodo_facturacion", "id_periodo_consumo", "consumo",
        "metodo_calculo", "tipo_consumo", "calificacion", "fecha_registro",
    ],
    "midas_datos_investigacion_consumo_bronze": [
        "servicio_suscrito", "tipo_consumo", "id_periodo_consumo", "solicitud_investigacion",
        "estado_investigacion", "estado_investigacion_desc", "fecha_registro",
    ],
}

print("VALIDACIÓN CAMPO A CAMPO")
print("=" * 88)
for tabla, esperadas in ESQUEMAS_REALES.items():
    if not existe(tabla):
        print(f"⚪ {tabla}: no existe todavía")
        continue
    actuales = spark.table(f"{PREFIJO}.{tabla}").columns
    faltantes = [c for c in esperadas if c not in actuales]
    extras = [c for c in actuales if c not in esperadas]
    estado = "✅" if not faltantes else "❌"
    print(f"{estado} {tabla}")
    if faltantes:
        print(f"   Faltan: {', '.join(faltantes)}")
    if extras:
        print(f"   Extras: {', '.join(extras)}")


---
# 5. Las tablas, una por una

A partir de aquí recorremos cada tabla. La estructura de cada sección es siempre la misma:

> **Qué es** en lenguaje de negocio → **De dónde sale** en Oracle → **Columnas que importan**
> → **Celda para explorar**

---
## 5.1 · `midas_ordenes_calidad_pendientes_bronze` — LA RAÍZ 🌳

**Qué es.** El punto de partida de toda la cadena. Cada fila es una **orden de calidad
pendiente**: un caso que un analista tiene que revisar.

**De dónde sale.** `or_order` + `or_order_activity`, con este filtro:

```sql
WHERE oa.task_type_id = 883        -- "órdenes de calidad"
  AND o.order_status_id <> 12      -- que no estén anuladas
  AND o.LEGALIZATION_DATE is null  -- y que sigan pendientes
```

> ### ★ El detalle más importante de todo el pipeline
> **Esta query NO filtra por actividad.** Trae *todas* las órdenes de calidad, de todos los
> casos de uso. Por eso en la columna `actividad` conviven `1019` (Caso 1),
> `993` (Caso 2) y varias más.
>
> **Consecuencia de arquitectura:** el Caso 2 **no necesitó un job nuevo**. Comparte la
> misma raíz y la misma cadencia que el Caso 1. La separación entre casos vive
> **aguas abajo** (en Silver, filtrando por `actividad`), nunca en Oracle. Esto está
> elevado a invariante del proyecto.

**Columnas clave:** `id_orden` · `servicio_suscrito` · `contrato` · `instalacion` ·
`actividad` · `estado_orden`

In [ ]:
df_ordenes = explorar("midas_ordenes_calidad_pendientes_bronze")

In [ ]:
# 🔍 EXPLORA — ¿cuántas órdenes hay de cada caso de uso?
# Esta consulta responde en segundos "¿qué volumen real tiene el Caso 2?",
# que es un dato útil para dimensionar el trabajo del agente.
if df_ordenes is not None:
    display(
        df_ordenes.groupBy("actividad")
                  .agg(F.count("*").alias("ordenes"))
                  .orderBy(F.desc("ordenes"))
    )
    print("Busca '993' (variación significativa) y '1019' (acueducto/alcantarillado).")

---
## 5.2 · `midas_datos_basicos_producto_bronze` — la ficha del servicio

**Qué es.** La ficha de identificación de un servicio suscrito: qué servicio es, de quién,
dónde queda, en qué estado está.

**De dónde sale.** `servsusc` + `pr_product` + `ab_address` + `suscripc` + `ge_subscriber`,
con los catálogos resueltos *inline* (`estacort`, `categori`, `subcateg`, `plansusc`,
`servicio`).

**Parámetro de entrada:** `:address_id` (la instalación).

**Columnas que el analista usa todo el tiempo:**

| Columna | Cómo lo dice el analista | Por qué importa |
|---|---|---|
| `estado_corte` | *"que el estado de corte sea facturable"* | **Paso 0 de todos los casos** |
| `categoria` / `subcategoria` | *"reviso la categoría de todos los SS"* | Detectar contratos con categorías mixtas |
| `plan_facturacion` | — | Identifica **áreas comunes** (7, 463) y **macromedidor** (986) |
| `fecha_instalacion` | *"el servicio es nuevo o es reinstalación"* | ⚠ cuidado: una reinstalación se ve igual que un servicio nuevo |
| `ciclo` | *"miro el ciclo"* | 1-20 metro · 101-123 regional · 24 especial |

In [ ]:
df_basicos = explorar("midas_datos_basicos_producto_bronze")

In [ ]:
# 🔍 EXPLORA — el plan de facturación esconde dos casos especiales.
# Busca los planes de ÁREAS COMUNES y el 986 (macromedidor).
perfil("midas_datos_basicos_producto_bronze", "plan_facturacion", top=20)

---
## 5.3 · `midas_datos_lecturas_producto_bronze` — el corazón del Caso 2 ❤️

**Qué es.** Los últimos **8 periodos** de lecturas del medidor. Es la pestaña que el
analista abre primero y donde pasa la mayor parte del tiempo.

**De dónde sale.** `lectelme` (lecturas) + `conssesu` (consumo facturado) + `elemmedi`
(el medidor) + `obselect` (observaciones del lector).

**Parámetro de entrada:** `:servicio_suscrito`.

### Las columnas críticas y qué significan

| Columna | Oracle | Cómo lo dice el analista |
|---|---|---|
| `observacion_lectura` | `LEEMOBLE` → `obselect` | ★ *"al lector no le criticó"* → valor `0` |
| `medidor` | `ELMECODI` | ★ *"aquí hay una serie nueva"* → cambió el medidor |
| `lectura_anterior` / `lectura_actual` | `LEEMLEAN` / `LEEMLETO` | *"el medidor viene devolviendo"* si baja |
| `consumo_calculado` | `(LEEMLETO−LEEMLEAN) × LEEMFAME` | Consumo por diferencia de lecturas |
| `consumo_facturado` | `COSSCOCA` con `COSSMECC=4` | ★ **lo que efectivamente se cobra** |
| `limite_inferior` / `limite_superior` | `LEEMLIIN` / `LEEMLISU` | *"está entre los límites"* |
| `constante` | atributo `5000058` | ★ Central para el caso de reactiva |
| `digitos_medidor` | `ELMENUDC` | ★ Detecta la "vuelta" falsa del medidor |
| `tipo_consumo` | `TIPOCONS.TCONCODI/TCONDESC` (descripción); `tipocons` conserva `LEEMTCON` crudo | ★★ **activa vs reactiva** — ¡ojo con esto! |
| `pno` | subquery `COSSMECC=17` | Marca de pérdida no operacional |

> ### ⚠️ Dos trampas que conviene conocer desde el principio
>
> **1. `consumo_calculado` ≠ `consumo_facturado`.** El primero es la resta de lecturas; el segundo
> es lo que se facturó. Divergen legítimamente en varios casos (medidor cambiado,
> áreas comunes, consumo estimado). No asumas que deben coincidir.
>
> **2. Los límites NO son un criterio de decisión.** En las sesiones grabadas hay órdenes
> que cierran *"sin ajuste"* estando **por encima** del límite, otras estando **por debajo**,
> y el propio analista advierte: *"como el límite inferior es cero, siempre va a estar entre
> los límites"*. Úsalos como **señal**, nunca como regla.

In [ ]:
df_lecturas = explorar("midas_datos_lecturas_producto_bronze")

In [ ]:
# 🔍 EXPLORA — la observación del lector.
# El valor "0-SIN CAUSA NI OBSERVACIÓN" es el que el analista lee como
# "no le criticó al lector", y es el discriminador de normalidad más usado.
# Se midió que ~73% de las lecturas están en 0.
perfil("midas_datos_lecturas_producto_bronze", "observacion_lectura", top=20)

### ⚠️⚠️ Verificación crítica: ¿se conserva la energía reactiva?

En energía, un mismo medidor produce **dos filas por periodo**:

| tipo_consumo | Qué mide |
|---|---|
| `3-ENERGÍA ACTIVA` | La energía que consume el cliente |
| `6-ENERGÍA REACTIVA` | La energía reactiva (se cobra solo la que excede el 50% de la activa) |

**Si alguna tabla dedujera por `(servicio_suscrito, id_periodo_consumo)` sin incluir
`tipo_consumo` en la llave, la reactiva se perdería** — y con ella los dos casos de mayor
impacto económico del Caso 2 (se observaron cobros erróneos de ~$29 M y ~$36 M que el
analista tuvo que corregir).

La celda siguiente lo verifica.

In [ ]:
# VERIFICACIÓN 1 — ¿hay más de un tipo de consumo por (servicio, periodo)?
# Si el resultado tiene filas, la reactiva SÍ se está conservando. ✅
# Si sale vacío en un ambiente con energía, hay que revisar la PK de la tabla. ⚠️
if df_lecturas is not None:
    conteo = (df_lecturas
              .groupBy("servicio_suscrito", "id_periodo_consumo")
              .agg(F.countDistinct("tipo_consumo").alias("tipos_distintos"),
                   F.collect_set("tipo_consumo").alias("cuales"))
              .filter(F.col("tipos_distintos") > 1))

    n = conteo.count()
    if n > 0:
        print(f"✅ OK — {n:,} combinaciones (servicio, periodo) con más de un tipo de consumo.")
        print("   La activa y la reactiva conviven correctamente.\n")
        display(conteo.limit(10))
    else:
        print("⚠️  No se encontró ningún (servicio, periodo) con más de un tipo de consumo.")
        print("   Puede ser normal si en este ambiente no hay órdenes de energía,")
        print("   PERO si las hay, revisa el DDL y la PK de la tabla: la reactiva")
        print("   podría estarse perdiendo en la deduplicación.")

---
## 5.4 · `midas_datos_consumos_producto_bronze` — el detalle del cálculo

**Qué es.** Cada forma en que se calculó el consumo del mes. Un mismo periodo puede tener
varias filas: una estimación, una corrección, y la que finalmente se facturó.

**De dónde sale.** `conssesu` + `mecacons` (métodos) + `calivaco` (calificaciones).

### La regla de oro

> **De todos los métodos de cálculo, solo `4-Consumo facturado` es el que se cobra.**
> Los demás son estimaciones, correcciones o ajustes intermedios. La columna `consumo` de
> la tabla de lecturas ya viene filtrada por este método.

| Método | Qué es |
|---|---|
| `1-Consumo medido` | Lectura real |
| `2-Consumo corregido` | Ajuste (suele venir en negativo) |
| `3-Consumo estimado` | Estimación del sistema |
| **`4-Consumo facturado`** | ★ **El único que se cobra** |
| `5-Consumo recuperado` | Recuperación |
| `17-Consumo recuperado por pérdida` | Marca de pérdida no operacional |
| `19-Consumo de área común` | Propiedad horizontal |
| `21-Consumo calculado ajustado` | Ajuste tras una revisión |

### Dos columnas con mucho valor oculto

- **`funcion_calculo`** — el proceso que generó el consumo. Dos valores son huellas
  digitales muy útiles:
  - `[P_SOLICITUD_DE_INVESTIGACION]` → el consumo está **en investigación**
  - `[FMLPN] - ESTACOPN` → viene de una **pérdida no operacional**
- **`calificacion`** — el veredicto del sistema. Tiene 30+ valores, y varios de ellos
  **son literalmente el texto con que el analista cierra la orden** (por ejemplo
  `5097-MEDIDOR CONFORME CALIBRACIÓN`).

In [ ]:
df_consumos = explorar("midas_datos_consumos_producto_bronze")

In [ ]:
# 🔍 EXPLORA — los métodos de cálculo. Fíjate en la proporción del método 4.
perfil("midas_datos_consumos_producto_bronze", "metodo_calculo", top=15)

In [ ]:
# 🔍 EXPLORA — las calificaciones. Varias son textos de cierre del analista.
perfil("midas_datos_consumos_producto_bronze", "calificacion", top=25)

In [ ]:
# 🔍 EXPLORA — las funciones de cálculo. Busca [P_SOLICITUD_DE_INVESTIGACION].
perfil("midas_datos_consumos_producto_bronze", "funcion_calculo", top=20)

---
## 5.5 · `midas_datos_cuentas_cobro_bronze` y `midas_datos_detalle_cargos_bronze`

**Qué son.** La factura y su desglose. La cuenta de cobro es el total del mes; los cargos
son las líneas que lo componen.

**De dónde salen.** `cuencobr` (cuentas) y `cargos` + `concepto` + `procesos` (detalle).

### La columna que delata el "Caso 17 — Otros cobros"

`valor_recuperado` es la suma de cargos cuyo **periodo de consumo es distinto** al de la
cuenta. Cuando tiene valor, significa que están cobrando meses anteriores — el analista
lo lee como *"está recuperando"*.

### La regla del Caso 17, verificada con datos reales

En un contrato observado, la cuenta pasó de ~$120.000 a **$703.715**. Pero el consumo
eran apenas **162 kWh**. La diferencia estaba en cargos que **no son de consumo**:

| Concepto | Programa | Unidades | Valor |
|---|---|---|---|
| `90-CONSUMO ACTIVA` | `5-Generar cargos recurrentes` | 162 | $118.665 |
| `1308-CONTROL PÉRDIDAS-CON IVA` | **`-26-Fénix`** | 1 | **$344.708** |
| `1307-MANTENIMIENTO-CON IVA` | **`-26-Fénix`** | 1 | $126.855 |

> ★ **La columna `programa` identifica el sistema que originó el cargo.** Fíjate en los
> códigos negativos (`-26-Fénix`, `-51-Sistema control pérdidas energía`): permiten separar
> "cargos de consumo" de "otros cobros" **sin depender del catálogo de conceptos**.

In [ ]:
df_cuentas = explorar("midas_datos_cuentas_cobro_bronze")
print("\n" + "="*70 + "\n")
df_cargos  = explorar("midas_datos_detalle_cargos_bronze")

In [ ]:
# 🔍 EXPLORA — los programas que generan cargos. Busca los códigos negativos.
perfil("midas_datos_detalle_cargos_bronze", "programa", top=20)

---
## 5.6 · `midas_datos_ordenes_previa_critica_bronze` y `midas_datos_cometarios_ordenes_bronze`

**Qué son.** Las órdenes de revisión previas y **sus comentarios**. Aquí está lo que el
técnico encontró en terreno.

> ⚠️ La tabla de comentarios tiene un **typo histórico** en el nombre: dice
> `cometarios`, no `comentarios`. Está documentado como deuda técnica; no lo corrijas
> por tu cuenta porque hay código que depende del nombre actual.

### El comentario es el campo más valioso y el más difícil

De las sesiones grabadas con analistas salieron **tres formatos distintos** de comentario:

**1. Prosa técnica con números embebidos** — hay que leerlo:
```
**CAMBIO MEDIDOR** LECTURA RETIRO 36241-35863= 378  SE INSTALA NUEV LECT 0-730= 730
TOTAL 1108 * KTE .8715 = 965.622
```

**2. Registro clave-valor serializado** — se puede *parsear*:
```
!INF_LECTURAS=Lectura tomada_431
!INF_CUADRILLA=... Aumento de Uso: hace 2 meses ... Se revisó con geófono, no existe fuga
!CODIGO_RESPUESTA=803 !ESTADO_ORDEN=E4044
;VARIACIÓN EN EL NIVEL DE UTILIZACIÓN POR MAYOR USO     ← ¡el texto de cierre, literal!
```

**3. Instrucción imperativa** — el agente debe **obedecerla**, no solo leerla:
```
NO DEJAR EN INVESTIGACION, NO PROMEDIAR. PARA EL CICLO DE MARZO SE DEBE COBRAR
TODO EL CONSUMO ... FAVOR COBRARLA TODA CON EL FACTOR DE CONSUMO
```

> ★ **Regla de precedencia descubierta:** cuando el comentario termina con una frase que
> coincide con el catálogo de cierres, **esa es la justificación**. Se observó en 3 contratos.

In [ ]:
df_criticas = explorar("midas_datos_ordenes_previa_critica_bronze")
print("\n" + "="*70 + "\n")
df_coment   = explorar("midas_datos_cometarios_ordenes_bronze", n=3)

In [ ]:
# 🔍 EXPLORA — mira comentarios reales. Cambia el número para ver más.
# Fíjate en cuál de los tres formatos corresponde cada uno.
if df_coment is not None:
    col_com = [c for c in df_coment.columns if "coment" in c.lower()]
    if col_com:
        filas = df_coment.select(*col_com).limit(5).collect()
        for i, fila in enumerate(filas, 1):
            print(f"\n--- Comentario {i} " + "-"*50)
            for c in col_com:
                if fila[c]:
                    print(str(fila[c])[:900])

### ⚠️ Verificación: ¿entran las órdenes de DECISIÓN DEL ANALISTA?

Existe una actividad llamada **`7400027 - ORDEN DECISIÓN ANALISTA`** (tipo de trabajo
`10038`). Su comentario **es la justificación final que escribió el analista**, y suele
referenciar la orden de investigación que resuelve:

```
ID OPEN 648518072 — Reparó fuga imperceptible detectada por: USUARIO
```

**Por qué esto es tan valioso:** significa que las decisiones humanas están registradas de
forma estructurada en Oracle, con fecha y comentario. Es decir, **ya existe el conjunto de
datos etiquetado** para entrenar y evaluar al agente.

**Brecha confirmada en el repo:** `QUERY_ORDENES_CRITICA_PEVIA` limita la primera rama a
`activity_id = 102010`; sus otras dos ramas filtran una lista de `task_type_id` que
incluye `10037` pero no `10038`. Por tanto, el notebook no debe asumir que
`7400027/10038` está presente en esta Bronze. La celda siguiente lo comprueba contra el
ambiente y deja visible la brecha.


In [ ]:
# VERIFICACIÓN 2 — ¿está llegando la ORDEN DECISIÓN ANALISTA (7400027 / 10038)?
if df_criticas is not None:
    col_act = [c for c in df_criticas.columns if "activid" in c.lower()]
    if col_act:
        c = col_act[0]
        print(f"Actividades presentes en la tabla (columna '{c}'):\n")
        display(df_criticas.groupBy(c).agg(F.count("*").alias("n")).orderBy(F.desc("n")))

        hay = df_criticas.filter(F.col(c).contains("7400027")).count()
        print("\n" + "="*70)
        if hay > 0:
            print(f"✅ SÍ llegan: {hay:,} órdenes de decisión del analista.")
            print("   Este es el ground truth para entrenar y evaluar el agente.")
        else:
            print("⚠️  NO se encontraron órdenes con actividad 7400027.")
            print("   Revisa los filtros de QUERY_ORDENES_CRITICA_PEVIA:")
            print("   la primera rama filtra activity_id = 102010, y la lista de")
            print("   task_type incluye 10037 pero quizá no 10038.")
            print("   Si se confirma, es probablemente la brecha de mayor valor pendiente.")

---
## 5.7 · `midas_datos_detalle_solicitudes_bronze` — **NUEVA (A1)** 🆕

**Qué es.** Los trámites que el cliente radicó: reclamos, reconexiones, suspensiones,
reinstalaciones, PQR.

**De dónde sale.** `mo_packages` + `mo_motive` + `ps_package_type` + catálogos.
Volumen medido: **6.38 millones de registros en 6 meses**.

**Por qué el Caso 1 no la necesitaba.** Comparar acueducto contra alcantarillado no
depende de trámites del cliente. El Caso 2 sí: 5 de sus casuísticas dependen de esta tabla.

> **Nota histórica:** esta tabla **ya existía** en Databricks pero estaba *huérfana* —
> fuera del plano de control, sin gobierno. El trabajo del Caso 2 la **adoptó**.

### Los tipos de solicitud que resuelven casos

| Código | Descripción | Qué caso resuelve |
|---|---|---|
| `300` | Reconexión por Pago | *"bajo consumo porque hace poco tuvo reconexión"* |
| `56` | Suspensión por no Pago | Marca el inicio del periodo suspendido |
| `42` | Reinstalación de Producto | ★ Distingue **reinstalación** de **servicio nuevo** |
| `15` | Retiro por No Pago | Antecede a la reinstalación |
| `288` | Gestión Administrativa de PNO | Pérdidas no operacionales |
| `289` | Aprobación de Ajustes de Facturación | PQR / fallo de la SSPD |
| `100207` | Solicitud de Investigación de Consumo | Investigación |

> ★ **Regla computable verificada con datos:** en un contrato real, la suspensión (`56`)
> fue el **5 de febrero** y la reconexión (`300`) el **18 de febrero** → **13 días sin
> servicio** dentro del periodo facturado. El consumo cayó de 26.663 a 8.216. La caída
> queda explicada por aritmética, sin necesidad de interpretar texto.

In [ ]:
df_solic = explorar("midas_datos_detalle_solicitudes_bronze")

In [ ]:
# 🔍 EXPLORA — los tipos de solicitud. Busca los 7 códigos de la tabla de arriba.
if df_solic is not None:
    col_tipo = [c for c in df_solic.columns
                if "package_type" in c.lower() or "tipo_solicitud" in c.lower()]
    if col_tipo:
        perfil("midas_datos_detalle_solicitudes_bronze", col_tipo[0], top=25)
    else:
        print("Columnas disponibles:", ", ".join(df_solic.columns))

---
## 5.8 · `midas_datos_servicios_contrato_bronze` — **NUEVA (A2)** 🆕 ★★

**Qué es.** Todos los servicios suscritos que cuelgan del mismo contrato — no solo el de
la orden.

**De dónde sale.** La misma consulta de `servsusc` usada por datos básicos, filtrada por `SESUSUSC` (el contrato) mediante `:p_contrato`.
Cardinalidad medida: **~4.6 servicios por contrato**.

**Esta es la tabla más importante del delta del Caso 2.**

### Por qué

El analista **razona a nivel de contrato**. Dos ejemplos reales de las sesiones grabadas:

> **Contrato 1299905** — La orden es de **energía**. El analista no encuentra explicación
> ahí, abre el **agua del mismo contrato**, y encuentra una revisión que dice *"aumento de
> uso... ocupó el local... funciona lavadero"*. Cierra la energía como *recién ocupado*.

> **Contrato 3716338** — Idéntico patrón: la justificación de la energía está en el
> comentario de la revisión del agua.

Sin esta tabla, esas órdenes **no se pueden resolver**.

Además: en **6 contratos observados** las categorías **difieren dentro del mismo contrato**
(gas comercial, agua residencial estrato 3). El documento original de casos de uso decía
que "deben ser iguales" — la evidencia dice que la heterogeneidad es **la norma**.

**Decisión de diseño:** se traen **todos** los SS, incluidos los retirados. La vigencia se
filtra en Silver, no en Bronze. Bronze es réplica fiel.

⚠️ **Impacto de rendimiento:** A2 produce el roster y A3 ejecuta el historial por cada SS encontrado (~4.6 en promedio). A3 está acotada a 6 meses.
Conviene monitorear `filas_leidas` en `midas_log_cargas`.

In [ ]:
# A2 — usa la salida ya obtenida de QUERY_SERVICIOS_CONTRATO.
TABLA_A2 = "midas_datos_servicios_contrato_bronze"
_resultados_directos = globals().get("RESULTADOS_DESTINO", {})
df_serv_contrato = _resultados_directos.get(TABLA_A2, pd.DataFrame())

if df_serv_contrato.empty:
    print("⚪ A2 no devolvió filas. Ejecuta primero la sección Oracle o revisa el contrato ancla.")
else:
    mostrar_pd(
        df_serv_contrato,
        f"A2 · QUERY_SERVICIOS_CONTRATO ({len(df_serv_contrato)} filas desde Oracle)",
    )

In [ ]:
# Nombre definitivo verificado en el seed de midas_control_cargas.
TABLA_A2 = "midas_datos_servicios_contrato_bronze"
df_serv_contrato = explorar(TABLA_A2)


In [ ]:
# Inventario opcional del esquema: útil para detectar tablas no gobernadas por este seed.
try:
    todas = spark.sql(f"SHOW TABLES IN {PREFIJO}")
    display(todas.filter(F.col("tableName").startswith("midas_")).orderBy("tableName"))
except Exception as e:
    print("No se pudo listar el esquema:", e)


In [ ]:
# 🔍 EXPLORA — ¿cuántos servicios tiene cada contrato?
# El valor medido en la Fase 1 fue ~4.6 en promedio.
if df_serv_contrato is not None and "contrato" in df_serv_contrato.columns:
    display(
        df_serv_contrato.groupBy("contrato")
                        .agg(F.count("*").alias("num_servicios"))
                        .orderBy(F.desc("num_servicios"))
                        .limit(20)
    )
    prom = (df_serv_contrato.groupBy("contrato").count()
                            .agg(F.avg("count").alias("promedio")).collect()[0]["promedio"])
    print(f"\nPromedio de servicios por contrato: {prom:.2f}")

---
## 5.9 · `midas_datos_consumos_contrato_bronze` — **NUEVA (A3)** 🆕

**Qué es.** El historial de consumo de **los servicios hermanos** del contrato.

**De dónde sale.** La misma `conssesu` de siempre, pero parametrizada por **cualquier** SS
del contrato, no solo el de la orden. Depende de A2 (primero hay que saber cuáles son).

**Por qué es una tabla aparte y no una ampliación de la existente.** Porque
`midas_datos_consumos_producto_bronze` tiene su clave definida sobre el SS **de la orden**.
Meter ahí los servicios hermanos mezclaría dos semánticas distintas en la misma tabla y
rompería el contrato con el agente del Caso 1.

**Qué habilita:** poder decir *"los otros servicios también disminuyeron"* (Caso 13) o
*"el gas venía vacío y ya lo ocuparon"* (Caso 7).

In [ ]:
# A3 — usa el consolidado de QUERY_CONSUMOS_CONTRATO ejecutado para los SS hermanos.
TABLA_A3 = "midas_datos_consumos_contrato_bronze"
_resultados_directos = globals().get("RESULTADOS_DESTINO", {})
df_cons_contrato = _resultados_directos.get(TABLA_A3, pd.DataFrame())

if df_cons_contrato.empty:
    print("⚪ A3 no devolvió filas para los SS del contrato seleccionados.")
else:
    mostrar_pd(
        df_cons_contrato,
        f"A3 · QUERY_CONSUMOS_CONTRATO ({len(df_cons_contrato)} filas desde Oracle)",
    )

In [ ]:
# Nombre definitivo verificado en el seed de midas_control_cargas.
TABLA_A3 = "midas_datos_consumos_contrato_bronze"
df_cons_contrato = explorar(TABLA_A3)


---
## 5.10 · `midas_datos_investigacion_consumo_bronze` — **NUEVA (A4)** 🆕

**Qué es.** El registro de consumos que quedaron **en investigación**.

**De dónde sale.** `PE_INVEST_CONSUM` — la query vigente proyecta 7 columnas y limita la historia a 6 meses.

**En la Bronze, el código crudo se llama `estado_investigacion` (origen `INVEST_CONS_STATE_ID`) y su etiqueta se llama `estado_investigacion_desc`:**

| Valor | Significado |
|---|---|
| `1` | **EN INVESTIGACIÓN** (abierta) |
| `2` | IMPUTABLE AL CLIENTE |
| `3` | IMPUTABLE A LA EMPRESA |

> Esto corrigió un supuesto previo que creía el `2` terminal. **`2` y `3` son
> resoluciones, no cierre.**

### ⚠️ El detalle del join que estuvo a punto de salir mal

El join correcto es por **`servicio_suscrito` + `id_periodo_consumo`**, no por
`id_periodo_facturacion`. Se verificó midiendo: **21 valores en común contra 0**.
Sin esa comprobación, el join habría salido vacío **en silencio**.

### El flag de investigación tiene TRES fuentes

| # | Fuente | Dónde vive | Valoración |
|---|---|---|---|
| 1 | `INVEST_CONS_STATE_ID` | **esta tabla (A4)** | El registro del proceso |
| 2 | Solicitud tipo `100207` | **solicitudes (A1)** | El trámite radicado |
| 3 | `funcion_calculo = [P_SOLICITUD_DE_INVESTIGACION]` + `calificacion = 5055` | **consumos (ya existía)** | ★ **La más fiable** |

**Recomendación:** construir el flag sobre la fuente 3, que está en la fila del propio
consumo y **no requiere extracción nueva**. Usar 1 y 2 como enriquecimiento.

In [ ]:
# A4 — usa el consolidado de QUERY_INVESTIGACION_CONSUMO ejecutado por SS.
TABLA_A4 = "midas_datos_investigacion_consumo_bronze"
_resultados_directos = globals().get("RESULTADOS_DESTINO", {})
df_invest = _resultados_directos.get(TABLA_A4, pd.DataFrame())

if df_invest.empty:
    print("⚪ A4 no devolvió consumos en investigación para la muestra seleccionada.")
else:
    mostrar_pd(
        df_invest,
        f"A4 · QUERY_INVESTIGACION_CONSUMO ({len(df_invest)} filas desde Oracle)",
    )
    if "estado_investigacion" in df_invest.columns:
        distribucion_estado = (
            df_invest["estado_investigacion"]
            .value_counts(dropna=False)
            .rename_axis("estado_investigacion")
            .reset_index(name="frecuencia")
        )
        display(distribucion_estado)


In [ ]:
# Nombre definitivo y columnas verificados en queries.py y en el seed.
TABLA_A4 = "midas_datos_investigacion_consumo_bronze"
df_invest = explorar(TABLA_A4)
if df_invest is not None:
    perfil(TABLA_A4, "estado_investigacion")


---
## 5.11 · `midas_dim_estado_corte_facturable_bronze` — **NUEVA (dimensión)** 🆕

**Qué es.** Una **matriz de decisión**: para cada combinación de *estado de corte* ×
*servicio*, dice si es **facturable** (S/N).

**De dónde sale.** `confesco` × `servicio`. Se carga como `QUERY_FULL_OVERWRITE`, es
decir, se regenera completa cada día y **no depende de ninguna orden**.

### Por qué esta es la única dimensión que sobrevivió

El diseño inicial contemplaba **9 tablas de dimensión**. Se redujeron a una, y el
razonamiento vale la pena entenderlo:

> Las queries de MIDAS ya resuelven los catálogos **inline**:
> `(SELECT escocodi||'-'||escodesc FROM estacort WHERE escocodi = sesuesco) AS estado_corte`.
> Como los códigos **ya llegan traducidos** a Bronze, ocho de las nueve dimensiones eran
> redundantes.
>
> Esta sobrevive porque **no es una etiqueta, es una regla**. El agente la consulta para
> decidir, necesita **todas las combinaciones posibles** (no solo las que aparecen en los
> datos del día), y el negocio pidió que fuera dinámica.

### ⚠️ Una advertencia que sale del análisis de las sesiones grabadas

*Facturable* y *analizable* **no son lo mismo**. Se observaron 6 de los 11 códigos:

| Código | Descripción | ¿Detiene el análisis? |
|---|---|---|
| `1` | Conexión | No |
| `4` | Orden suspensión total | **No** — se observó analizándose |
| `5` | Suspensión total | **No**, pero **cambia el texto de cierre** |
| `95` / `110` / `970` | Retiros | Sí — son SS históricos |

Conviene confirmar en el taller si la matriz `confesco` captura esa distinción o si hace
falta una segunda columna.

In [ ]:
# Dimensión — usa QUERY_DIM_ESTADO_CORTE_FACTURABLE ejecutada directamente.
TABLA_DIM = "midas_dim_estado_corte_facturable_bronze"
_resultados_directos = globals().get("RESULTADOS_DESTINO", {})
df_dim = _resultados_directos.get(TABLA_DIM, pd.DataFrame())

if df_dim.empty:
    print("⚪ La query de la dimensión no devolvió filas.")
else:
    mostrar_pd(
        df_dim.head(30),
        f"Dimensión facturable · origen Oracle ({len(df_dim)} filas recuperadas)",
    )

In [ ]:
# Nombre definitivo verificado en el seed de midas_control_cargas.
TABLA_DIM = "midas_dim_estado_corte_facturable_bronze"
df_dim = explorar(TABLA_DIM, n=30)


---
# 6. Cómo se unen las tablas

Este es el mapa de llaves. **Casi todo se une por `servicio_suscrito`.**

```
                  midas_ordenes_calidad_pendientes_bronze
                  (LA RAÍZ — trae 993, 1019 y las demás)
                                  │
        ┌─────────────────────────┼──────────────────────────┐
        │ instalacion             │ servicio_suscrito        │ contrato
        ▼                         ▼                          ▼
  datos_basicos          ┌────────────────────┐      servicios_contrato  🆕
    _producto            │ lecturas_producto  │              │
        │                │ consumos_producto  │              │ servicio_suscrito
        │                │ cuentas_cobro      │              ▼
        │                │ detalle_cargos     │      consumos_contrato   🆕
        │                │ ordenes_previa_    │
        │                │   critica ─────────┼── id_orden ──► cometarios_ordenes
        │                │ detalle_solicitudes│ 🆕
        │                │ investigacion_     │ 🆕
        │                │   consumo          │
        │                └────────────────────┘
        │
        └── estado_corte + servicio ──► dim_estado_corte_facturable  🆕
```

### Las llaves compuestas que hay que respetar

| Unión | Llave |
|---|---|
| lecturas ↔ consumos | `servicio_suscrito` + `id_periodo_consumo` + **`tipo_consumo`** |
| órdenes de crítica ↔ comentarios | `id_orden` |
| investigación ↔ consumos | `servicio_suscrito` + **`id_periodo_consumo`** ⚠ *no* `id_periodo_facturacion` |
| servicios del contrato ↔ básicos | `contrato` |
| cuentas de cobro ↔ cargos | **`id_cuenta_cobro`** (ambas tablas también conservan `servicio_suscrito`) |

> ⚠️ **Nunca olvides `tipo_consumo`** al unir lecturas con consumos en energía. Si lo
> omites, mezclarás la activa con la reactiva y los números no cuadrarán.

In [ ]:
# 🔍 EXPLORA — un recorrido completo desde una orden hasta sus datos.
# Cambia el número para explorar otra orden.
if df_ordenes is not None:
    fila = df_ordenes.limit(1).collect()
    if fila:
        SS       = fila[0]["servicio_suscrito"]
        CONTRATO = fila[0]["contrato"]
        print(f"Explorando el servicio suscrito {SS} (contrato {CONTRATO})\n")

        if df_lecturas is not None:
            print("--- Sus últimas lecturas ---")
            display(df_lecturas.filter(F.col("servicio_suscrito") == SS)
                               .orderBy(F.desc("id_periodo_consumo")).limit(10))

---
# 7. Los casos de uso, traducidos a consultas

Aquí está el pago de todo lo anterior: las reglas del analista, convertidas en SQL.

Estas cuatro son **determinísticas** — no necesitan interpretar texto libre. Salieron del
análisis de 5 sesiones grabadas con analistas y **todas están verificadas contra datos
reales**.

### 7.1 · Caso 3/4 — Cambio de medidor

**Cómo lo dice el analista:** *"aquí sí se ve que hay una serie nueva, o sea que el medidor
lo cambiaron"*.

**La regla:** si en un mismo periodo hay **más de un medidor** para el mismo servicio y
tipo de consumo, hubo cambio de medidor. El sistema cobra la suma de ambos.

**Verificado 4 veces.** Por ejemplo: `239.252 + 174.492 = 413.744`, y ese total coincide
exactamente con las unidades del cargo de consumo.

In [ ]:
# REGLA 1 — Detección de cambio de medidor
if df_lecturas is not None:
    cambio_medidor = (df_lecturas
        .groupBy("servicio_suscrito", "id_periodo_consumo", "tipo_consumo")
        .agg(F.countDistinct("medidor").alias("n_medidores"),
             F.collect_set("medidor").alias("medidores"),
             F.sum("consumo_facturado").alias("consumo_total"))
        .filter(F.col("n_medidores") > 1)
        .orderBy(F.desc("consumo_total")))

    n = cambio_medidor.count()
    print(f"Periodos con más de un medidor (cambio de medidor): {n:,}\n")
    if n:
        display(cambio_medidor.limit(20))
        print("→ En estos casos el cierre esperado es:")
        print("  'Sin ajuste - Estableciendo promedios con un medidor nuevo'")

### 7.2 · Caso 23b — Vuelta falsa del medidor ⚠️ alto impacto

**Cómo lo dice el analista:** *"la lectura anterior era 2 y ahora es 1... el sistema piensa
que dio la vuelta... está cobrando 99 mil unidades"*.

**Qué pasa:** cuando la lectura actual es **menor** que la anterior, el sistema asume que
el medidor completó una vuelta y calcula `(10^dígitos − anterior) + actual`. Con un medidor
de 5 dígitos eso da **99.999 unidades**.

**El impacto real observado: ~$36 millones** que el analista tuvo que corregir a mano.

**La regla:** `consumo_calculado < 0` **y** `consumo_facturado` cercano a `10^digitos_medidor`.

In [ ]:
# REGLA 2 — Vuelta falsa del medidor
if df_lecturas is not None:
    cols = df_lecturas.columns
    if "digitos_medidor" in cols and "consumo_calculado" in cols:
        vuelta = (df_lecturas
            .withColumn("umbral", F.pow(F.lit(10), F.col("digitos_medidor").cast("int")) * 0.9)
            .filter((F.col("consumo_calculado") < 0) & (F.col("consumo_facturado") > F.col("umbral")))
            .select("servicio_suscrito", "id_periodo_consumo", "tipo_consumo",
                    "digitos_medidor", "lectura_anterior", "lectura_actual",
                    "consumo_calculado", "consumo_facturado"))

        n = vuelta.count()
        print(f"Casos de posible vuelta falsa del medidor: {n:,}\n")
        if n:
            display(vuelta.limit(20))
            print("→ Cierre esperado: 'CON AJUSTE - Se corrige consumo por lectura menor'")
            print("⚠️  Estos casos generan cobros de decenas de millones. Revisar con prioridad.")
        else:
            print("✅ No se detectaron vueltas falsas en este ambiente.")
    else:
        print("⚠️  Faltan columnas digitos_medidor o consumo_calculado.")

### 7.3 · Caso 22/23a — Energía reactiva y constante errónea ⚠️ alto impacto

**Cómo lo dice el analista:** *"este es de esos casos en que no le cambian la constante.
Cuando arreglaron la activa, no se dieron cuenta que al modificar la activa dañaba la
reactiva"*.

**La fórmula de la reactiva** (derivada de los datos y **verificada 5 veces** en 3
contratos distintos):

```
consumo_reactiva_facturado = MAX(0, consumo_calc_reactiva − 0.5 × consumo_activa_facturado)
```

En Colombia solo se cobra la energía reactiva que **excede el 50% de la activa**. Cuando
alguien corrige la constante de la activa pero no la de la reactiva, ese piso del 50% se
desploma y **casi toda la reactiva pasa a facturarse**.

> ⚠️ El factor `0.5` está pendiente de confirmación formal con negocio, aunque la
> aritmética encaja exactamente en 5 periodos.

In [ ]:
# REGLA 3 — Verificación de la fórmula de la reactiva
if df_lecturas is not None:
    activa = (df_lecturas.filter(F.col("tipocons").cast("string") == "3")
              .select(F.col("servicio_suscrito").alias("ss"),
                      F.col("id_periodo_consumo").alias("per"),
                      F.col("consumo_facturado").alias("consumo_activa")))

    reactiva = (df_lecturas.filter(F.col("tipocons").cast("string") == "6")
                .select("servicio_suscrito", "id_periodo_consumo",
                        F.col("consumo_calculado").alias("calc_reactiva"),
                        F.col("consumo_facturado").alias("consumo_reactiva")))

    comparacion = (reactiva
        .join(activa,
              (reactiva.servicio_suscrito == activa.ss) &
              (reactiva.id_periodo_consumo == activa.per), "inner")
        .withColumn("esperado",
                    F.greatest(F.lit(0.0),
                               F.col("calc_reactiva") - 0.5 * F.col("consumo_activa")))
        .withColumn("diferencia", F.abs(F.col("consumo_reactiva") - F.col("esperado")))
        .select("servicio_suscrito", "id_periodo_consumo", "consumo_activa",
                "calc_reactiva", "consumo_reactiva", "esperado", "diferencia"))

    n = comparacion.count()
    if n:
        print(f"Pares activa/reactiva encontrados: {n:,}")
        print("Si 'diferencia' es ~0, la fórmula se confirma en tus datos.\n")
        display(comparacion.orderBy(F.desc("diferencia")).limit(20))
    else:
        print("No se encontraron pares activa/reactiva.")
        print("Puede ser normal si en este ambiente no hay órdenes de energía.")

### 7.4 · Caso 17 — "Otros cobros"

**Cómo lo dice el analista:** *"no hay nada anormal en las lecturas, no tiene reclamo, pero
aquí sí se nota que el cobro se subió cinco veces lo que gasta normalmente"*.

**La regla:** el valor de la cuenta sube pero **las unidades del concepto de consumo no**.
La diferencia está en cargos que no son de consumo (mantenimiento, control de pérdidas,
calibración), identificables por la columna `programa`.

In [ ]:
# REGLA 4 — Cargos que no son de consumo
if df_cargos is not None and "programa" in df_cargos.columns:
    no_consumo = (df_cargos
        .filter(F.col("programa").rlike(r"^-"))   # los programas negativos vienen de otros sistemas
        .groupBy("programa", "concepto")
        .agg(F.count("*").alias("n_cargos"),
             F.sum("valor").alias("valor_total"))
        .orderBy(F.desc("valor_total")))

    n = no_consumo.count()
    print(f"Conceptos generados por sistemas externos (programa con código negativo): {n:,}\n")
    if n:
        display(no_consumo.limit(25))
        print("→ Cierre esperado cuando estos dominan la cuenta: 'Sin ajuste - Otros cobros'")

---
# 8. Lista de verificación del pipeline

Antes de confiar en estos datos para construir el agente, conviene confirmar estos puntos.
La celda siguiente los revisa todos de una vez.

In [ ]:
print("="*72)
print("  VERIFICACIÓN DEL MODELO BRONZE DEL CASO 2")
print("="*72)

TABLAS_ESPERADAS = [
    ("midas_ordenes_calidad_pendientes_bronze",  "Caso 1 · raíz"),
    ("midas_datos_basicos_producto_bronze",      "Caso 1"),
    ("midas_datos_lecturas_producto_bronze",     "Caso 1"),
    ("midas_datos_consumos_producto_bronze",     "Caso 1"),
    ("midas_datos_cuentas_cobro_bronze",         "Caso 1"),
    ("midas_datos_detalle_cargos_bronze",        "Caso 1"),
    ("midas_datos_ordenes_previa_critica_bronze","Caso 1"),
    ("midas_datos_cometarios_ordenes_bronze",    "Caso 1 (typo histórico)"),
    ("midas_datos_detalle_solicitudes_bronze",   "Caso 2 · A1"),
    (TABLA_A2 or "midas_datos_servicios_contrato_bronze",   "Caso 2 · A2"),
    (TABLA_A3 or "midas_datos_consumos_contrato_bronze",    "Caso 2 · A3"),
    (TABLA_A4 or "midas_datos_investigacion_consumo_bronze","Caso 2 · A4"),
    (TABLA_DIM or "midas_dim_estado_corte_facturable_bronze","Caso 2 · dimensión"),
    ("midas_parametros",                         "Caso 2 · parámetros"),
]

print("\n1) EXISTENCIA DE TABLAS\n" + "-"*72)
faltan = []
for t, grupo in TABLAS_ESPERADAS:
    if existe(t):
        print(f"  ✅ {t:<52} [{grupo}]")
    else:
        print(f"  ❌ {t:<52} [{grupo}]")
        faltan.append(t)

print("\n2) COMPROBACIONES CRÍTICAS\n" + "-"*72)

# a) reactiva
if df_lecturas is not None:
    multi = (df_lecturas.groupBy("servicio_suscrito", "id_periodo_consumo")
             .agg(F.countDistinct("tipo_consumo").alias("t"))
             .filter(F.col("t") > 1).count())
    print(f"  {'✅' if multi else '⚠️ '} Activa/reactiva separadas: {multi:,} periodos con >1 tipo")

# b) orden decisión analista
if df_criticas is not None:
    ca = [c for c in df_criticas.columns if "activid" in c.lower()]
    if ca:
        d = df_criticas.filter(F.col(ca[0]).contains("7400027")).count()
        print(f"  {'✅' if d else '⚠️ '} Orden decisión analista (7400027): {d:,} registros")

# c) actividad 993
if df_ordenes is not None:
    n993 = df_ordenes.filter(F.col("actividad").contains("993")).count()
    print(f"  {'✅' if n993 else '⚠️ '} Órdenes del Caso 2 (993): {n993:,}")

# d) constante y digitos
if df_lecturas is not None:
    for c in ("constante", "digitos_medidor"):
        if c in df_lecturas.columns:
            nulos = df_lecturas.filter(F.col(c).isNull()).count()
            tot   = df_lecturas.count()
            pct   = 100 * nulos / tot if tot else 0
            print(f"  {'✅' if pct < 50 else '⚠️ '} {c}: {pct:.1f}% nulos")
        else:
            print(f"  ❌ {c}: la columna no existe")

print("\n" + "="*72)
if faltan:
    print(f"  Faltan {len(faltan)} tablas. Puede ser normal si no se ha desplegado todo.")
else:
    print("  Todas las tablas esperadas están presentes.")
print("="*72)

---
# 9. Glosario

| Término | Significado |
|---|---|
| **FLEX** | El sistema Oracle de facturación de EPM (heredero de Open Smartflex). Fuente primaria |
| **SS** | *Servicio Suscrito*. El servicio de un predio (el agua, la energía…). La entidad central |
| **Contrato** | La cuenta que agrupa varios SS. **El nivel al que razona el analista** |
| **Instalación** | El predio físico |
| **Bronze / Silver / Gold** | Capas del modelo *medallion*: cruda / modelada / consumo |
| **`FULL_CHAINED`** | Carga encadenada: la query necesita un parámetro de una tabla anterior |
| **`QUERY_FULL_OVERWRITE`** | Carga autocontenida: se materializa completa cada día |
| **`job_name`** | Discriminador del plano de control. Para MIDAS Bronze es `midas_bronze` |
| **Caso 1 · actividad `1019`** | Diferencia acueducto/alcantarillado. Comparación cerrada |
| **Caso 2 · actividad `993`** | **Variación significativa de consumo**. 23 casuísticas |
| **`task_type = 883`** | El filtro que define "orden de calidad" en la raíz |
| **`102010`** | Actividad *Analizar consumo crítico* |
| **`7400027` / `10038`** | ★ *Orden decisión analista* — la etiqueta de referencia del agente |
| **PNO** | Pérdida No Operacional (fraude) |
| **Áreas comunes** | Propiedad horizontal. Plan `7` / `463`, método de cálculo `19` |
| **Macromedidor** | Medidor general de un transformador. Plan de facturación `986` |
| **HIDRO / GDE / GASPAR / FENIX** | Sistemas aprovisionadores (agua / energía / gas / medidores de gas) |
| **PEAJF / FEFP** | Módulos de OPEN donde el analista ejecuta los ajustes |

---

## Para seguir

| Documento | Qué contiene |
|---|---|
| `MIDAS_Caso2_CasosDeUso_CONSOLIDADO.md` | Los 23 casos paso a paso, con las cuatro capas de trazabilidad |
| `MIDAS_Caso2_Diccionario_Datos.xlsx` | 96 campos mapeados · catálogos · reglas · hallazgos |
| `MIDAS_Comparacion_Bronze_Caso1_vs_Caso2.md` | Por qué se agregaron las tablas nuevas + diagrama |

---

*Cuaderno de solo lectura. No modifica datos.*